## Phase 0: Configuration and Field Mapping

Issues and to-dos:
* Now
    * CUIs! Ugh!
    * English FSN --> Long Common Name - Done?
    * Compare a couple of LOINCs to search.loinc.org, OHDSI, LOINC FHIR, and OCL's 2.71
    * Common_Test_rank "0" - should be '-' instead of 0
    * Linguistic variants - en --> en-GB
* 

Known missing pieces:
* Community Mappings
* be sure to include mapping guidance. Examples:
        https://loinc.org/60567-5/ Term Description – this is “Comment” in the 2.71 repo version, and Joe is now modeling as a “Description” for 2.80 and 2.81
        https://loinc.org/74774-1 - Reference Information -> Mapping Guidance – this term should not be used in the US at all

In [2]:
### LOINC FILE DISCOVERY AND COMPILATION ###

import os
import shutil
import re
from pathlib import Path

# Set the source directory where LOINC files are located (in various subfolders)
source_loinc_directory = r"C:/Users/jamlung/Documents/LOINC/Loinc_2.80"  # UPDATE THIS PATH

# Define the target files we need to find and their expected names
required_files = {
    'Loinc.csv': ['Loinc.csv', 'LOINC.csv', 'loinc.csv'],
    'Part.csv': ['Part.csv', 'PartLink.csv', 'PART.csv', 'part.csv'],
    'AnswerList.csv': ['AnswerList.csv', 'ANSWERLIST.csv', 'answerlist.csv'],
    'PanelsAndForms.csv': ['PanelsAndForms.csv', 'PANELSANDFORMS.csv', 'panelsandforms.csv'],
    'LoincAnswerListLink.csv': ['LoincAnswerListLink.csv', 'LoincAnswerListLink.csv', 'loincanswer listlink.csv'],
    'MapTo.csv': ['MapTo.csv', 'MAPTO.csv', 'mapto.csv'],
    'ComponentHierarchyBySystem.csv': ['ComponentHierarchyBySystem.csv', 'COMPONENTHIERARCHYBYSYSTEM.csv', 'componenthierarchybysystem.csv']
}

def is_linguistic_variant_file(filename, filepath):
    """
    Determines if a file is likely a linguistic variant based on flexible patterns.
    """
    # Skip known main LOINC files
    main_file_patterns = [
        'loinc.csv', 'part.csv', 'answerlist.csv', 'panelsandforms.csv',
        'loincanswer', 'mapto.csv', 'componenthierarchy'
    ]
    
    filename_lower = filename.lower()
    if any(pattern in filename_lower for pattern in main_file_patterns):
        return False
    
    # Pattern 1: Files starting with 2-letter language code (xx.csv or xxYY.csv)
    pattern1 = r'^[a-z]{2}([A-Z]{2})?\.csv$'
    if re.match(pattern1, filename):
        return True
        
    # Pattern 2: Files starting with language-country code (e.g., esMX_something.csv)
    pattern2 = r'^[a-z]{2}[A-Z]{2}[_-].*\.csv$'
    if re.match(pattern2, filename):
        return True
        
    # Pattern 3: Files in directories suggesting linguistic variants
    path_indicators = ['linguistic', 'variant', 'translation', 'locale', 'lang', 'i18n', 'l10n']
    if any(indicator in filepath.lower() for indicator in path_indicators):
        return True
        
    # Pattern 4: Files with linguistic indicators in name
    name_indicators = ['linguistic', 'variant', 'translation', 'locale']
    if any(indicator in filename_lower for indicator in name_indicators):
        return True
        
    # Pattern 5: More flexible language code detection
    # Check if first 2-4 characters look like language codes
    if len(filename) >= 5:  # Minimum: xx.csv (5 chars)
        # Check for 2-letter language code followed by delimiter
        if (filename[:2].islower() and filename[:2].isalpha() and 
            filename[2] in ['.', '_', '-']):
            return True
            
        # Check for 4-letter language-country code followed by delimiter
        if (len(filename) >= 7 and  # Minimum: xxYY.csv (7 chars)
            filename[:2].islower() and filename[:2].isalpha() and
            filename[2:4].isupper() and filename[2:4].isalpha() and
            filename[4] in ['.', '_', '-']):
            return True
            
    return False

def find_and_compile_loinc_files(source_dir, target_dir='Input'):
    """
    Searches for LOINC CSV files in subdirectories and copies them to a target folder.
    Also handles linguistic variant files specially.
    """
    # Create target directory
    os.makedirs(target_dir, exist_ok=True)
    
    print(f"🔍 Searching for LOINC files in: {source_dir}")
    print(f"📁 Target directory: {target_dir}")
    
    found_files = {}
    missing_files = []
    
    # Search for each required file
    for target_name, possible_names in required_files.items():
        file_found = False
        
        # Walk through all subdirectories
        for root, dirs, files in os.walk(source_dir):
            for filename in files:
                if filename in possible_names:
                    source_path = os.path.join(root, filename)
                    target_path = os.path.join(target_dir, target_name)
                    
                    # Copy file to target directory
                    shutil.copy2(source_path, target_path)
                    found_files[target_name] = source_path
                    print(f"✅ Found and copied: {filename} → {target_name}")
                    file_found = True
                    break
            
            if file_found:
                break
        
        if not file_found:
            missing_files.append(target_name)
            print(f"❌ Not found: {target_name} (searched for: {', '.join(possible_names)})")
    
    # Handle linguistic variant files specially - preserve original names
    linguistic_variants_dir = os.path.join(target_dir, 'LinguisticVariants')
    os.makedirs(linguistic_variants_dir, exist_ok=True)
    
    linguistic_files_found = 0
    print(f"\n🌐 Searching for linguistic variant files (preserving original names)...")
    
    for root, dirs, files in os.walk(source_dir):
        for filename in files:
            if filename.endswith('.csv'):
                full_path = os.path.join(root, filename)
                
                if is_linguistic_variant_file(filename, full_path):
                    source_path = full_path
                    # Preserve original filename - no renaming
                    target_path = os.path.join(linguistic_variants_dir, filename)
                    
                    shutil.copy2(source_path, target_path)
                    linguistic_files_found += 1
                    print(f"🌐 Found linguistic variant: {filename} (from {os.path.relpath(root, source_dir)})")
    
    print(f"\n📊 Summary:")
    print(f"   Found {len(found_files)}/{len(required_files)} required files")
    print(f"   Found {linguistic_files_found} linguistic variant files")
    
    if missing_files:
        print(f"   Missing files: {', '.join(missing_files)}")
        print(f"   ⚠️  You may need to manually locate and copy these files")
    
    return found_files, missing_files, linguistic_files_found

def update_file_paths(target_dir='Input'):
    """
    Updates the file path variables to point to the compiled Input directory.
    """
    global loinc_csv_path, part_link_csv_path, answer_list_csv_path
    global linguistic_variants_path, panels_and_forms_csv_path
    global loinc_answer_list_link_csv_path, map_to_csv_path, component_hierarchy_file_path
    
    # Update all file paths to point to Input directory
    loinc_csv_path = os.path.join(target_dir, 'Loinc.csv')
    part_link_csv_path = os.path.join(target_dir, 'Part.csv')
    answer_list_csv_path = os.path.join(target_dir, 'AnswerList.csv')
    linguistic_variants_path = os.path.join(target_dir, 'LinguisticVariants')
    panels_and_forms_csv_path = os.path.join(target_dir, 'PanelsAndForms.csv')
    loinc_answer_list_link_csv_path = os.path.join(target_dir, 'LoincAnswerListLink.csv')
    map_to_csv_path = os.path.join(target_dir, 'MapTo.csv')
    component_hierarchy_file_path = os.path.join(target_dir, 'ComponentHierarchyBySystem.csv')
    
    print(f"\n🔄 Updated file paths to use {target_dir} directory:")
    print(f"   LOINC CSV: {loinc_csv_path}")
    print(f"   Part CSV: {part_link_csv_path}")
    print(f"   Answer List CSV: {answer_list_csv_path}")
    print(f"   Linguistic Variants: {linguistic_variants_path}")
    print(f"   Panels and Forms CSV: {panels_and_forms_csv_path}")
    print(f"   LOINC Answer List Link CSV: {loinc_answer_list_link_csv_path}")
    print(f"   Map To CSV: {map_to_csv_path}")
    print(f"   Component Hierarchy CSV: {component_hierarchy_file_path}")

# Check if source directory exists and run the compilation
if os.path.exists(source_loinc_directory):
    print("🚀 Starting LOINC file discovery and compilation...")
    found, missing, linguistic_count = find_and_compile_loinc_files(source_loinc_directory)
    
    if len(found) >= 4:  # Minimum files needed to proceed
        update_file_paths()
        print("\n✅ File compilation completed successfully!")
        print("   You can now proceed with the rest of the notebook.")
    else:
        print(f"\n❌ Compilation incomplete. Found only {len(found)} out of {len(required_files)} required files.")
        print("   Please check your source directory path and ensure LOINC files are present.")
        
else:
    print(f"❌ Source directory not found: {source_loinc_directory}")
    print("   Please update the 'source_loinc_directory' variable with the correct path.")
    print("   If you don't have LOINC files, the notebook will use demo/test data instead.")

🚀 Starting LOINC file discovery and compilation...
🔍 Searching for LOINC files in: C:/Users/jamlung/Documents/LOINC/Loinc_2.80
📁 Target directory: Input
✅ Found and copied: Loinc.csv → Loinc.csv
✅ Found and copied: Part.csv → Part.csv
✅ Found and copied: AnswerList.csv → AnswerList.csv
✅ Found and copied: PanelsAndForms.csv → PanelsAndForms.csv
✅ Found and copied: LoincAnswerListLink.csv → LoincAnswerListLink.csv
✅ Found and copied: MapTo.csv → MapTo.csv
✅ Found and copied: ComponentHierarchyBySystem.csv → ComponentHierarchyBySystem.csv

🌐 Searching for linguistic variant files (preserving original names)...
🌐 Found linguistic variant: deAT24LinguisticVariant.csv (from AccessoryFiles\LinguisticVariants)
🌐 Found linguistic variant: deDE15LinguisticVariant.csv (from AccessoryFiles\LinguisticVariants)
🌐 Found linguistic variant: elGR17LinguisticVariant.csv (from AccessoryFiles\LinguisticVariants)
🌐 Found linguistic variant: esAR7LinguisticVariant.csv (from AccessoryFiles\LinguisticVariant

In [3]:
### CONFIGURATION & DEMO VARIABLES ###

# Set the path for your input CSV files
loinc_csv_path = 'Input/Loinc.csv'
part_link_csv_path = 'Input/Part.csv'
answer_list_csv_path = 'Input/AnswerList.csv'
linguistic_variants_path = 'Input/LinguisticVariants'
panels_and_forms_csv_path = 'Input/PanelsAndForms.csv'
loinc_answer_list_link_csv_path = 'Input/LoincAnswerListLink.csv'
map_to_csv_path = 'Input/MapTo.csv'
component_hierarchy_file_path = "Input/ComponentHierarchyBySystem.csv"
config_path = 'UMLS_API_config.json'


# Set the output path for the transformed JSON files
output_folder = 'output'

# Set this to 1 or 2 to run the notebook with example data instead of your real CSVs.
# NOTE: The demo data below is for illustration; real-world data is much larger.
mode = 0  # 0 = full run with all data, 1 = test_mode with real subset of data up to 15 records per CSV, 2 = demo_mode with example data

In [4]:
### PACKAGE IMPORTS ###

# Import pandas for data manipulation and analysis
import pandas as pd

# Import numpy for numerical operations, often used for NaN values
import numpy as np

# Import json for saving to JSON Lines format
import json

#Import StringIO to handle in-memory text streams
from io import StringIO

# Import os for file and directory operations
import os

# Import re for regular expression operations
import re

# Import time for timing operations
import time

# For ignoring performance warnings from pandas
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

from typing import List, Dict, Set

In [5]:
# Example DataFrames for demo_mode (mode = 2)
# These are based on the CSV snippets you provided.
demo_loinc_df = pd.DataFrame({
    'LOINC_NUM': ['100000-9', '100001-7'],
    'COMPONENT': ['Health informatics pioneer and the father of LOINC', 'Health informatics pioneer and cofounder of LOINC'],
    'PROPERTY': ['Hx', 'LP431396-3'],
    'TIME_ASPCT': ['Pt', 'Pt'],
    'SYSTEM': ['^Patient', 'Ser'],
    'SCALE_TYP': ['Nar', 'Qn'],
    'SHORTNAME': ['Health Info Pioneer+Father of LOINC', 'Health Info Pioneer+Cofounder of LOINC'],
    'LONG_COMMON_NAME': ['Health informatics pioneer and the father of LOINC', 'Health informatics pioneer and cofounder of LOINC'],
    'STATUS': ['ACTIVE', 'ACTIVE']
})

demo_part_link_df = pd.DataFrame({
    'LoincNumber': ['100000-9', '100000-9', '100000-9', '100000-9', '100000-9'],
    'LongCommonName': ['Health informatics pioneer and the father of LOINC'] * 5,
    'PartNumber': ['LP431397-1', 'LP6817-3', 'LP6960-1', 'LP310005-6', 'LP7749-7'],
    'PartName': ['Health informatics pioneer and the father of LOINC', 'Hx', 'Pt', '^Patient', 'Nar'],
    'PartCodeSystem': ['http://loinc.org'] * 5,
    'PartTypeName': ['COMPONENT', 'PROPERTY', 'TIME', 'SYSTEM', 'SCALE'],
    'LinkTypeName': ['Primary'] * 5,
    'Property': ['http://loinc.org/property/COMPONENT', 'http://loinc.org/property/PROPERTY', 'http://loinc.org/property/TIME_ASPCT', 'http://loinc.org/property/SYSTEM', 'http://loinc.org/property/SCALE_TYP']
})

demo_answer_list_df = pd.DataFrame({
    'AnswerListId': ['LL1000-0', 'LL1000-0', 'LL1000-0', 'LL1001-8'],
    'AnswerListName': ['PhenX05_13_30D bread amt', 'PhenX05_13_30D bread amt', 'PhenX05_13_30D bread amt', 'PhenX05_14_30D freq amts'],
    'AnswerListOID': ['1.3.6.1.4.1.12009.10.1.165'] * 3 + ['1.3.6.1.4.1.12009.10.1.166'],
    'ExtDefinedYN': ['N'] * 4,
    'AnswerStringId': ['LA13825-7', 'LA13838-0', 'LA13892-7', 'LA6270-8'],
    'SequenceNumber': [1, 2, 3, 1],
    'DisplayText': ['1 slice or 1 dinner roll', '2 slices or 2 dinner rolls', 'More than 2 slices or 2 dinner rolls', 'Never']
})

# Linguistic variant CSV demo data 
esMX_data = """LOINC_NUM,COMPONENT,PROPERTY,TIME_ASPCT,SYSTEM,SCALE_TYP,METHOD_TYP,CLASS,SHORTNAME,LONG_COMMON_NAME,RELATEDNAMES2,LinguisticVariantDisplayName
33512-5,Color,Tipo,Punto temporal,XXX,Nominal,,,,Color: XXX : Punto temporal: Tipo: Nominal:,,
24355-0,Panel macroscópico de análisis de orina,-,Punto temporal,Orina,-,,,,Panel macroscópico de análisis de orina: Orina : Punto temporal: -: -:,,
10003-2,Duración de la onda R. derivación III,Tiempo,Punto temporal,Corazón,Cuantitativo,EKG,,,Duración de la onda R. derivación III:Corazón :Punto temporal:Tiempo:Cuantitativo:EKG,,
10006-5,Duración de la onda R. derivación V3,Tiempo,Punto temporal,Corazón,Cuantitativo,EKG,,,Duración de la onda R. derivación V3:Corazón :Punto temporal:Tiempo:Cuantitativo:EKG,,
10007-3,Duración de la onda R. derivación V4,Tiempo,Punto temporal,Corazón,Cuantitativo,EKG,,,Duración de la onda R. derivación V4:Corazón :Punto temporal:Tiempo:Cuantitativo:EKG,,
10024-8,Duración de la onda R 'plomo AVR,Tiempo,Punto temporal,Corazón,Cuantitativo,EKG,,,Duración de la onda R 'plomo AVR:Corazón :Punto temporal:Tiempo:Cuantitativo:EKG,,
1004-1,"Test de antiglobulina directo, reactivo específico del complemento",Presencia o umbral,Punto temporal,Eritrocitos,Ordinal,,,,"Test de antiglobulina directo, reactivo específico del complemento: Eritrocitos : Punto temporal: Presencia o umbral: Ordinal:",,
10060-2,Amplitud de onda S. Conduzca AVR,Elpot,Punto temporal,Corazón,Cuantitativo,EKG,,,Amplitud de onda S. Conduzca AVR:Corazón :Punto temporal:Elpot:Cuantitativo:EKG,,
10063-6,Amplitud de onda S. derivación III,Elpot,Punto temporal,Corazón,Cuantitativo,EKG,,,Amplitud de onda S. derivación III:Corazón :Punto temporal:Elpot:Cuantitativo:EKG,,
10280-6,Número de modelo del proveedor,Tipo,Punto temporal,Tubo de cobre,Nominal,,,,Número de modelo del proveedor:Tubo de cobre :Punto temporal:Tipo:Nominal:,,
10445-5,CD11c Ag,Presencia o umbral,Punto temporal,Tejido y frotis,Ordinal,Mancha inmune,,,CD11c Ag: Tejido y frotis : Punto temporal: Presencia o umbral: Ordinal: Mancha inmune,,
10455-4,Xilosa ^30 M después de 25 g de xilosa VO,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Xilosa : Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
10539-5,glipizida,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,glipizida: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
10547-8,Primidona + FENobarbital,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Primidona + FENobarbital: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
10550-2,Temazepam,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Temazepam: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
1099-1,K sub p super sub a Ab,Presencia o umbral,Punto temporal,Suero o Plasma,Ordinal,,,,K sub p super sub a Ab: Suero o Plasma : Punto temporal: Presencia o umbral: Ordinal:,,
10995-9,Neomicina,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Neomicina: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
11001-5,Pirazinamida,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Pirazinamida: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,"""

etEE_data = """LOINC_NUM,COMPONENT,PROPERTY,TIME_ASPCT,SYSTEM,SCALE_TYP,METHOD_TYP,CLASS,SHORTNAME,LONG_COMMON_NAME,RELATEDNAMES2,LinguisticVariantDisplayName
93488-5,Guanidinoatsetaat,SCnc,Pt,Vereplekk,Qn,,CHEM,,,Aine kontsentratsioon Juhuslik Kvantitatiivne Veri,
93505-6,Heptakarboksüülporfüriin I,SRat,24 tunni,U,Qn,,CHEM,,,Aine määr Kvantitatiivne Uriin,
93729-2,Beeta-2-mikroglobuliin/kreatiniin,Suhe,24 tunni,U,Qn,,CHEM,,,Kvantitatiivne Uriin,
93748-2,Fibriini monomeerid,MCnc,Pt,PPP,Qn,IA,COAG,,,Juhuslik Kvantitatiivne Trombotsüütidevaene plasma,
95073-3,Histoplasma capsulatum antigeen,MCnc,Pt,BalF,Qn,IA,MICRO,,,Juhuslik Kvantitatiivne,
95074-1,Bakterid,PrThr,Pt,BalF,Ord,Valgusmikroskoopia,MICRO,,,Järgarvuline Juhuslik,
89481-6,Gentamütsiin,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,
92255-9,Metitsilliin,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,
92242-7,Pürasiinamiid,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,
96635-8,HLA-C,Tüüp,Pt,B/Tis^doonor,Nom,,HLA,,,Juhuslik Kude Veri Veri või koematerjal,
95563-3,16-alfahüdroksüdehüdroepiandrosteroon,MRat,24 tunni,U,Qn,,CHEM,,,Kvantitatiivne Uriin,
95593-0,25-hüdroksükaltsiferool,MCnc,Pt,cB,Qn,,CHEM,,,Juhuslik Kapillaarne veri Kvantitatiivne,
95114-5,Insuliin^2 tundi pärast sööki,Acnc,Pt,S/P,Qn,,CHAL,,,Juhuslik Kvantitatiivne Plasma Seerum Seerum või plasma,
93773-0,11-deoksükortisool,SCnc,Pt,Sal,Qn,,CHEM,,,Aine kontsentratsioon Juhuslik Kvantitatiivne Sülg,
93838-1,Histoplasma capsulatum antikehad.IgM,Acnc,Pt,CSF,Qn,IA,MICRO,,,Immuunglobuliin M Juhuslik Kvantitatiivne Liikvor,
94255-7,Kaltsium,MCnc,Pt,Sw,Qn,,CHEM,,,Higi Juhuslik Kvantitatiivne,
94256-5,Magneesium,MCnc,Pt,Sw,Qn,,CHEM,,,Higi Juhuslik Kvantitatiivne,
94270-6,Ubikinoon 10,SCnt,Pt,WBC,Qn,,CHEM,,,Ainehulga sisaldus Juhuslik Kvantitatiivne Leukotsüüdid,
95543-5,Aspergillus terreus antikehad.IgG,PrThr,Pt,S,Ord,,ALLERGY,,,Immuunglobuliin G Järgarvuline Juhuslik Seerum,
95527-8,Tsütomegaloviirus antikehad.IgG,PrThr,Pt,Sal,Ord,IA,MICRO,,,Immuunglobuliin G Järgarvuline Juhuslik Sülg,
95688-8,Dengue viiruse 1.+ 2.+ 3.+ 4. tüüp antikehad.IgM,PrThr,Pt,XXX,Ord,IA,MICRO,,,Immuunglobuliin M Järgarvuline Juhuslik Täpsustamata materjal,
93771-4,Kalprotektiin,MCnc,Pt,SynF,Qn,,CHEM,,,Juhuslik Kvantitatiivne Liigesevedelik sünoviaalvedelik,
95574-0,17-alfahüdroksüpregnanoloon,MRat,24 tunni,U,Qn,,CHEM,,,Kvantitatiivne Uriin,
95594-8,Kaltsidiool,MCnc,Pt,cB,Qn,,CHEM,,,Juhuslik Kapillaarne veri Kvantitatiivne,
95675-5,Kollapalaviku viirus antikehad.IgG,PrThr,Pt,XXX,Ord,IA,MICRO,,,Immuunglobuliin G Järgarvuline Juhuslik Täpsustamata materjal,
95719-1,Flaviviirus antikehad,PrThr,Pt,S/P,Ord,,MICRO,,,Järgarvuline Juhuslik Plasma Seerum Seerum või plasma,
95800-9,Immuunglobuliini vabad kerged ahelad.paneel,-,-,U,-,,PANEL.CHEM,,,Uriin,
95966-8,Aspergillus glaucus antikehad.IgE,Acnc,Pt,S,Qn,,ALLERGY,,,Immuunglobuliin E Juhuslik Kvantitatiivne Seerum,
96043-5,Uratsüül,MCnc,Pt,S/P,Qn,,CHEM,,,Juhuslik Kvantitatiivne Plasma Seerum Seerum või plasma,
96108-6,Klofasimiin,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,
96111-0,Linetsoliid,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,"""

frBE_data = """LOINC_NUM,COMPONENT,PROPERTY,TIME_ASPCT,SYSTEM,SCALE_TYP,METHOD_TYP,CLASS,SHORTNAME,LONG_COMMON_NAME,RELATEDNAMES2,LinguisticVariantDisplayName
103631-8,Natalizumab,Concentration de masse,Temps ponctuel,Sérum,Ordinal,IA,Médicaments et produits toxiques,,,,
106016-9,Bactéries,Présence ou identité,Temps ponctuel,Pénis,Nominal,Culture,Microbiologie,,,Verge,
106033-4,Bactéries,Présence ou identité,Temps ponctuel,Pus,Nominal,Culture anaérobique,Microbiologie,,,,
106034-2,Champignon,Présence ou identité,Temps ponctuel,Pus,Nominal,Culture,Microbiologie,,,,
103648-2,Hormone folliculo-stimulante^4 h post dose hormone de libération des gonadotrophines,Concentration arbitraire,Temps ponctuel,Sérum/Plasma,Quantitatif,,Tests de provocation,,,"4 h post dose GNRH FSH Gn-RF, Gonadotrophines-releasing factor",
103685-4,Citalopram,Concentration de masse,Temps ponctuel,Urine,Quantitatif,LC/MS/MS,Médicaments et produits toxiques,,,,
103806-6,Note,Observation,Temps ponctuel,Contact téléphonique,Document,Oncologie,DOC.CLINRPT,,,,
103830-6,Gabapentine,PrThr,Temps ponctuel,Méconium,Ordinal,,Médicaments et produits toxiques,,,,
103834-8,Norbuprenorphine,PrThr,Temps ponctuel,Méconium,Ordinal,,Médicaments et produits toxiques,,,,
103839-7,Phentermine,PrThr,Temps ponctuel,Méconium,Ordinal,,Médicaments et produits toxiques,,,,
103958-5,Ofloxacine,Susceptibilité,Temps ponctuel,Isolat,Ordinal,Génotypage,Sensibilité aux antibiotiques,,,,
104133-4,Éthanol,PrThr,Temps ponctuel,Gaz expiré,Ordinal,,Médicaments et produits toxiques,,,,
104181-3,Toxine du clostridium tetani,PrThr,Temps ponctuel,Sérum/Plasma,Ordinal,Test biologique sur souris,Microbiologie,,,,
104183-9,Adénovirus ADN,PrThr,Temps ponctuel,Spécimen conjonctival,Ordinal,Sonde avec amplification de la cible,Microbiologie,,,,
104196-1,Créatine/Créatinine,Ratio de substance,Temps ponctuel,Sang sur papier filtre,Quantitatif,,Chimie,,,,
104234-0,Atomoxétine,Concentration de masse,Temps ponctuel,Urine,Quantitatif,Confirmé,Médicaments et produits toxiques,,,,
104237-3,Zopiclone,Concentration de masse,Temps ponctuel,Urine,Quantitatif,Confirmé,Médicaments et produits toxiques,,,,
104419-7,Legionella sp Ac^1er échantillon,Titre,Temps ponctuel,Sérum,Ordinal,IA,Microbiologie,,,Anticorps Echantillon.1,
104457-7,Virus varicelle-zona Anticorps.IgA,PrThr,Temps ponctuel,LCR,Ordinal,IA,Microbiologie,,,Anticorps VZV,
104459-3,Virus varicelle-zona Anticorps.IgG,PrThr,Temps ponctuel,LCR,Ordinal,IA,Microbiologie,,,Anticorps VZV,"""


In [6]:
### FIELD MAPPING DATASET - LOINCs ###

# This dictionary maps LOINC CSV fields to their corresponding OCL Concept fields.
# Mappings are based on the provided PDF and the LOINC FHIR example.
loinc_to_ocl_mapping = {    
    # General Format: '[Loinc Field]':'[OCL field]'

    #Field Mappings
    'LOINC_NUM': ['id','extras.Code_in_Source'],
    'LONG_COMMON_NAME': ['names.Fully-Specified.en[1]','extras.LONG_COMMON_NAME'],
    'DisplayName': ['names.Display.en[1]','extras.DisplayName'],
    'SHORTNAME': ['names.Short.en[1]','extras.SHORTNAME'],
    'CONSUMER_NAME': ['names.Consumer.en[1]','extras.CONSUMER_NAME'],
    'SCALE_TYP': ['datatype','extras.SCALE_TYP'],
    'STATUS': ['retired','extras.STATUS'], # Retired translation: ACTIVE/TRIAL/DISCOURAGED → false, DEPRECATED → true
    'DefinitionDescription': ['description','extras.DEFINITION_DESCRIPTION'],

    # Mappings derived from LOINC FHIR example JSON
    # These will be stored as `extras` to align with the FHIR properties.
    'COMPONENT': 'extras.COMPONENT',
    'PROPERTY': 'extras.PROPERTY',
    'TIME_ASPCT': 'extras.TIME_ASPCT',
    'SYSTEM': 'extras.SYSTEM',
    'METHOD_TYP': 'extras.METHOD_TYP',
    'VersionFirstReleased':'extras.VersionFirstReleased',
    'VersionLastChanged':'extras.VersionLastChanged',
    'ORDER_OBS':'extras.ORDER_OBS',
    'HL7_FIELD_SUBFIELD_ID':'extras.HL7_FIELD_SUBFIELD_ID',
    'EXTERNAL_COPYRIGHT_NOTICE':'extras.EXTERNAL_COPYRIGHT_NOTICE',
    'SURVEY_QUEST_TEXT':'extras.SURVEY_QUEST_TEXT',
    'SURVEY_QUEST_SRC':'extras.SURVEY_QUEST_SRC',
    'UNITSREQUIRED':'extras.UNITSREQUIRED',
    'RELATEDNAMES2':'extras.RELATEDNAMES2',
    'EXTERNAL_COPYRIGHT_LINK': 'extras.EXTERNAL_COPYRIGHT_LINK',
    'ValidHL7AttachmentRequest': 'extras.ValidHL7AttachmentRequest',
    'CHNG_TYPE': 'extras.CHNG_TYPE',
    'STATUS_TEXT': 'extras.STATUS_TEXT',
    'STATUS_REASON': 'extras.STATUS_REASON',
    'PanelType': 'extras.PanelType',
    'CHANGE_REASON_PUBLIC': 'extras.CHANGE_REASON_PUBLIC',
    'COMMON_TEST_RANK': 'extras.COMMON_TEST_RANK',
    'AskAtOrderEntry': 'extras.AskAtOrderEntry',
    'AssociatedObservations': 'extras.AssociatedObservations',
    'EXAMPLE_UNITS': 'extras.EXAMPLE_UNITS',
    'EXMPL_ANSWERS': 'extras.EXMPL_ANSWERS',
    'EXAMPLE_UCUM_UNITS': 'extras.EXAMPLE_UCUM_UNITS',
    'HL7_ATTACHMENT_STRUCTURE': 'extras.HL7_ATTACHMENT_STRUCTURE',
    'COMMON_ORDER_RANK': 'extras.COMMON_ORDER_RANK',
    'FORMULA': 'extras.FORMULA',
    'CLASS': 'extras.CLASS',
    'CLASSTYPE': 'extras.CLASSTYPE'

}

# This dictionary contains the fixed values to be used in the transformation.
fixed_values_loinc = {
    'type': 'Concept',
    'concept_class': 'LOINC',
    'source': 'LOINC',
    'owner_type': 'Organization',
    'owner': 'Regenstrief',
    'extras.Code_Type': 'LOINC'
}

In [7]:
### FIELD MAPPING DATASET - LOINC Parts ###

loinc_part_to_ocl_mapping ={
    "PartNumber": ["id", "extras.Code_in_Source"],
    "PartTypeName": "extras.PartTypeName",
    "PartName": ["names.Fully-Specified.en[1]", "extras.LONG_COMMON_NAME"],
    "PartDisplayName": ["names.Display.en[1]", "extras.PartDisplayName"],
    'Status': ['retired','extras.STATUS'] # 'retired' translation: ACTIVE/TRIAL/DISCOURAGED → false, DEPRECATED → true
}

# This dictionary contains the fixed values to be used in the transformation.
fixed_values_loinc_parts = {
    'type': 'Concept',
    'concept_class': 'LOINC Part',
    'datatype': 'N/A',
    'source': 'LOINC',
    'owner_type': 'Organization',
    'owner': 'Regenstrief',
    'extras.Code_Type': 'LOINC Part'
}

In [8]:
### FIELD MAPPING DATASET - LOINC Answer Lists and Answers ###

# Answer Lists
answer_list_to_ocl_mapping = {
    "AnswerListId": ["id", "extras.Code_in_Source"],
    "AnswerListName": ["names.Fully-Specified.en[1]", "extras.AnswerListName"],
    "AnswerListOID": "extras.AnswerListOID",
    "ExtDefinedYN": "extras.ExtDefinedYN",
    "ExtDefinedAnswerListCodeSystem": "extras.ExtDefinedAnswerListCodeSystem",
    "ExtDefinedAnswerListLink": "extras.ExtDefinedAnswerListLink"
}

fixed_values_answer_list = {
    'type': 'Concept',
    'retired': False,
    "concept_class": "Answer List",
    "datatype": "N/A",
    "source": "LOINC",
    "owner_type": "Organization",
    "owner": "Regenstrief",
    "extras.Code_Type": "LOINC Answer List"
}

# Answers
answer_to_ocl_mapping = {
    "AnswerStringId": ["id", "extras.Code_in_Source"],
    "DisplayText": ["names.Display.en[1]", "extras.DisplayText"],
    "LocalAnswerCode": "extras.LocalAnswerCode",
    "LocalAnswerCodeSystem": "extras.LocalAnswerCodeSystem",
    "SequenceNumber": "extras.SequenceNumber",
    "ExtCodeId": "extras.ExtCodeId",
    "ExtCodeDisplayName": ["names.Fully-Specified.en[1]", "extras.ExtCodeDisplayName"],
    "ExtCodeSystem": "extras.ExtCodeSystem",
    "ExtCodeSystemVersion": "extras.ExtCodeSystemVersion",
    "ExtCodeSystemCopyrightNotice": "extras.ExtCodeSystemCopyrightNotice",
    "SubsequentTextPrompt": "extras.SubsequentTextPrompt",
    "Description": "extras.Description",
    "Score": "extras.Score"
}

fixed_values_answer = {
    'type': 'Concept',
    'retired': False,
    "concept_class": "LOINC Answer",
    "datatype": "N/A",
    "source": "LOINC",
    "owner_type": "Organization",
    "owner": "Regenstrief",
    "extras.Code_Type": "LOINC Answer"
}

In [9]:
#Transformation rules - specify fields to transform and their target values

transformation_rules =  [
    {
      "field": ["STATUS", "Status"],
      "transformations": {
        "DEPRECATED": True,
        "ACTIVE": False,
        "TRIAL": False,
        "DISCOURAGED": False
      },
      "target_field": "retired"
    }
  ]

In [10]:
### Mapping Configurations ###

# Define the base URL for OCL concepts
CONCEPT_URL_PREFIX = "/orgs/Regenstrief/sources/LOINC/concepts/{}/"

# Define the overall configuration dictionary
MAPPING_CONFIGS = {
    "Panel-to-Test": {
        "source_files": ["PanelsAndForms.csv"],
        "map_type": "Has Element",
        "field_mappings": [
            {"ocl_field": "type", "source_value": "Mapping"},
            {"ocl_field": "map_type", "source_value": "Has Element"},
            {"ocl_field": "from_concept_url", "source_field": "ParentLoinc", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "to_concept_url", "source_field": "Loinc", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "source", "source_value": "LOINC"},
            {"ocl_field": "owner_type", "source_value": "Organization"},
            {"ocl_field": "owner", "source_value": "Regenstrief"},
            {"ocl_field": 'extras["Sequence"]', "source_field": "SequenceInPanel"},
            {"ocl_field": 'extras["Required"]', "source_field": "Required"},
            {
                "ocl_field": 'extras["Cardinality"]',
                "source_field_min": "CardinalityMin",
                "source_field_max": "CardinalityMax",
                "rule": lambda min_val, max_val: f"{min_val}..{max_val}",
            },
            {"ocl_field": 'extras["Answer List Override"]', "source_field": "AnswerListIdOverride"},
            {"ocl_field": 'extras["Answer List Type Override"]', "source_field": "AnswerListTypeOverride"},
        ],
    },
    "Question-to-Answer": {
        "source_files": ["LoincAnswerListLink.csv", "AnswerList.csv"],
        "map_type": "Has Answer",
        "join_condition": {"left_on": "LoincAnswerListLink.AnswerListId", "right_on": "AnswerList.AnswerListId"},
        "field_mappings": [
            {"ocl_field": "type", "source_value": "Mapping"},
            {"ocl_field": "map_type", "source_value": "Has Answer"},
            {"ocl_field": "from_concept_url", "source_field": "LoincNumber", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "to_concept_url", "source_field": "AnswerStringId", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "source", "source_value": "LOINC"},
            {"ocl_field": "owner_type", "source_value": "Organization"},
            {"ocl_field": "owner", "source_value": "Regenstrief"},
            {"ocl_field": 'extras["Answer List ID"]', "source_field": "LoincAnswerListLink.AnswerListId"},
            {"ocl_field": 'extras["Answer List Type"]', "source_field": "LoincAnswerListLink.AnswerListLink Type"},
            {"ocl_field": 'extras["Sequence"]', "source_field": "AnswerList.SequenceNumber"},
            {"ocl_field": 'extras["Score"]', "source_field": "AnswerList.Score"},
            {"ocl_field": 'extras["Local Answer Code"]', "source_field": "AnswerList.LocalAnswerCode"},
        ],
    },
    "Ask at Order Entry": {
        "source_files": ["Loinc.csv"],
        "map_type": "Ask At Order Entry",
        "field_mappings": [
            {"ocl_field": "type", "source_value": "Mapping"},
            {"ocl_field": "map_type", "source_value": "Ask At Order Entry"},
            {"ocl_field": "from_concept_url", "source_field": "LOINC_NUM", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "to_concept_url", "source_field": "AskAtOrderEntry", "rule": lambda value: CONCEPT_URL_PREFIX.format(value) if pd.notna(value) else None},
            {"ocl_field": "source", "source_value": "LOINC"},
            {"ocl_field": "owner_type", "source_value": "Organization"},
            {"ocl_field": "owner", "source_value": "Regenstrief"},
        ],
    },
    "Code Evolution": {
        "source_files": ["MapTo.csv"],
        "map_type": "Map To",
        "field_mappings": [
            {"ocl_field": "type", "source_value": "Mapping"},
            {"ocl_field": "map_type", "source_value": "Map To"},
            {"ocl_field": "from_concept_url", "source_field": "LOINC", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "to_concept_url", "source_field": "MAP_TO", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "source", "source_value": "LOINC"},
            {"ocl_field": "owner_type", "source_value": "Organization"},
            {"ocl_field": "owner", "source_value": "Regenstrief"},
            {"ocl_field": 'extras["COMMENT"]', "source_field": "COMMENT"},
        ],
    },
    "Associated Observations": {
        "source_files": ["Loinc.csv"],
        "map_type": "Associated Observations",
        "field_mappings": [
            {"ocl_field": "type", "source_value": "Mapping"},
            {"ocl_field": "map_type", "source_value": "Associated Observations"},
            {"ocl_field": "from_concept_url", "source_field": "LOINC_NUM", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "to_concept_url", "source_field": "AssociatedObservations", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "source", "source_value": "LOINC"},
            {"ocl_field": "owner_type", "source_value": "Organization"},
            {"ocl_field": "owner", "source_value": "Regenstrief"},
        ],
    },
}

In [11]:
###  Hierarchy Configurations  ###

hierarchy_mapping = {
    
    #Field Mappings
    # PATH_TO_ROOT,SEQUENCE,IMMEDIATE_PARENT,CODE,CODE_TEXT

    'PATH_TO_ROOT': 'extras.PATH_TO_ROOT',
    'SEQUENCE': 'extras.HIERARCHY_SEQUENCE',
    'IMMEDIATE_PARENT': 'parent_concept', # This will become the parent concept URL
    'CODE': 'match-id' # This will be used to match with the LOINC or Part concept ID
    # 'CODE_TEXT' will be ignored
}

## Phase 1: Data Loading and Validation

In [12]:
# Data Loading and Validation - Concepts

def load_data():
    """Loads data based on the `mode` variable and validates fields against the mapping."""
    if mode == 2:
        loinc_df = demo_loinc_df
        part_link_df = demo_part_link_df
        answer_list_df = demo_answer_list_df
    elif mode == 1:
        # In test mode, we load a small subset of the real data
        try:
            loinc_df = pd.read_csv(loinc_csv_path, nrows=15, low_memory=False)
        except FileNotFoundError:
            loinc_df = pd.DataFrame()
            print(f"⚠️ Warning: The file '{loinc_csv_path}' was not found. LOINC validation will be skipped.")
            
        part_link_df = pd.read_csv(part_link_csv_path, nrows=15, low_memory=False)
        answer_list_df = pd.read_csv(answer_list_csv_path, nrows=15, low_memory=False)
    elif mode == 0:
        # In full run mode, we load the entire CSV files
        try:
            loinc_df = pd.read_csv(loinc_csv_path, low_memory=False)
        except FileNotFoundError:
            loinc_df = pd.DataFrame()
            print(f"⚠️ Warning: The file '{loinc_csv_path}' was not found. LOINC validation will be skipped.")
            
        part_link_df = pd.read_csv(part_link_csv_path, low_memory=False)
        answer_list_df = pd.read_csv(answer_list_csv_path, low_memory=False)
    else:
        print("Invalid mode selected. Please use 0, 1, or 2.")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    
    # ... (rest of the load_data function remains the same) ...
    # The validation logic below is not changed.

    # A helper function to create a DataFrame from the raw CSV data
    def load_csv_data(csv_data):
        return pd.read_csv(StringIO(csv_data))

    # This part needs to be changed. The code below should dynamically
    # load files from the linguistic_variants_path directory for modes 0 and 1.
    if mode in [0, 1]:
        print("Loading linguistic variant files...")
        linguistic_variant_dfs = []
        for filename in os.listdir(linguistic_variants_path):
            if filename.endswith('.csv'):
                filepath = os.path.join(linguistic_variants_path, filename)
                df = pd.read_csv(filepath, low_memory=False)
                linguistic_variant_dfs.append(df)
    else: # mode == 2
        # Use hardcoded data for demo mode
        esMX_df = load_csv_data(esMX_data)
        etEE_df = load_csv_data(etEE_data)
        frBE_df = load_csv_data(frBE_data)
        linguistic_variant_dfs = [esMX_df, etEE_df, frBE_df]
    
    print(f"Loaded {len(linguistic_variant_dfs)} linguistic variant files.")
    
    return loinc_df, part_link_df, answer_list_df, linguistic_variant_dfs

# Call the function to load all data
loinc_df, part_link_df, answer_list_df, linguistic_variant_dfs = load_data()

def load_and_preprocess_hierarchy_data(file_path, column_map):
    """
    Loads the ComponentHierarchyBySystem CSV, renames and selects columns
    based on the provided mapping, and handles initial data types.
    """
    try:
        df = pd.read_csv(file_path)
        
        # Select only the columns specified in the mapping.
        # The key represents the new column name, and the value is the old column name.
        df = df[list(column_map.keys())]
        
        # Rename the selected columns using the mapping.
        df = df.rename(columns=column_map)
        
        return df
    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found.")
        return None

# Load the data into a DataFrame.
hierarchy_df = load_and_preprocess_hierarchy_data(component_hierarchy_file_path, hierarchy_mapping)

if hierarchy_df is not None:
    print("Successfully loaded and renamed hierarchy data with the new mapping:")
    # print(hierarchy_df.head())

Loading linguistic variant files...
Loaded 20 linguistic variant files.
Successfully loaded and renamed hierarchy data with the new mapping:


In [13]:
# Adds linguistic variant file names, dynamically generated from the directory.

# Function to check if a file is a proper linguistic variant file
def is_proper_linguistic_variant_file(filepath):
    """
    Checks if a file is a proper linguistic variant file by:
    1. Checking if filename matches the pattern xxYY##LinguisticVariant.csv
    2. Checking if it has the expected columns
    """
    filename = os.path.basename(filepath)
    
    # Skip the index file
    if filename == 'LinguisticVariants.csv':
        return False
    
    # Check if filename matches the expected pattern
    # Pattern: xxYY##LinguisticVariant.csv where xx=language, YY=country, ##=number
    pattern = r'^[a-z]{2}[A-Z]{2}\d+LinguisticVariant\.csv$'
    if not re.match(pattern, filename):
        return False
    
    try:
        # Check if file has the expected columns
        df_sample = pd.read_csv(filepath, nrows=1)
        expected_columns = ['LOINC_NUM', 'COMPONENT', 'PROPERTY', 'TIME_ASPCT', 
                          'SYSTEM', 'SCALE_TYP', 'METHOD_TYP', 'CLASS']
        
        # Check if at least the core columns exist
        missing_columns = [col for col in expected_columns if col not in df_sample.columns]
        if missing_columns:
            print(f"⚠️  Skipping {filename}: missing columns {missing_columns}")
            return False
            
        return True
    except Exception as e:
        print(f"⚠️  Error checking {filename}: {e}")
        return False

# Function to create the Fully Specified Name (FSN) with error handling
def create_fsn(row):
    """
    Creates FSN from row data with proper error handling
    """
    try:
        # List of columns to check for FSN creation
        fsn_columns = ['COMPONENT', 'PROPERTY', 'TIME_ASPCT', 'SYSTEM', 'SCALE_TYP', 'METHOD_TYP', 'CLASS']
        
        # Check if all required columns exist
        missing_cols = [col for col in fsn_columns if col not in row.index]
        if missing_cols:
            # If missing critical columns, return None
            return None
            
        parts = [row[col] for col in fsn_columns]
        valid_parts = [str(part) for part in parts if pd.notna(part) and str(part).strip() != '']
        
        return ':'.join(valid_parts) if valid_parts else None
        
    except Exception as e:
        print(f"⚠️  Error creating FSN for row: {e}")
        return None

# Function to process a single linguistic variant file with error handling
def process_linguistic_file(file_path):
    """
    Process a single linguistic variant file and add the locale code
    """
    filename = os.path.basename(file_path)
    
    # Extract locale code from filename (first 4 characters: xxYY)
    if len(filename) >= 4:
        # Extract language (first 2 chars) and country (next 2 chars) separately
        language_code = filename[:2].lower()   # e.g., "es" from "esMX28LinguisticVariant.csv"
        country_code = filename[2:4].upper()   # e.g., "MX" from "esMX28LinguisticVariant.csv"
        locale_code = f"{language_code}-{country_code}"  # e.g., "es-MX"
    else:
        locale_code = filename[:2].lower()  # fallback to just language code
    
    print(f"Processing {filename} with locale code: {locale_code}")  # e.g., "fr-FR"
    
    try:
        df = pd.read_csv(file_path, low_memory=False)
        processed_rows = []
        
        # Process each row
        for index, row in df.iterrows():
            # Try to create FSN
            fsn = create_fsn(row)
            if fsn:
                processed_rows.append({
                    'LOINC_NUM': row['LOINC_NUM'], 
                    'LocaleCode': locale_code,
                    'LinguisticVariantName': fsn, 
                    'NameType': 'Fully-Specified'
                })
            
            # Process SHORTNAME if available
            if 'SHORTNAME' in row and pd.notna(row['SHORTNAME']) and str(row['SHORTNAME']).strip():
                processed_rows.append({
                    'LOINC_NUM': row['LOINC_NUM'], 
                    'LocaleCode': locale_code,
                    'LinguisticVariantName': row['SHORTNAME'], 
                    'NameType': 'Short'
                })
                
            # Process LONG_COMMON_NAME if available
            if 'LONG_COMMON_NAME' in row and pd.notna(row['LONG_COMMON_NAME']) and str(row['LONG_COMMON_NAME']).strip():
                processed_rows.append({
                    'LOINC_NUM': row['LOINC_NUM'], 
                    'LocaleCode': locale_code,
                    'LinguisticVariantName': row['LONG_COMMON_NAME'], 
                    'NameType': 'Display'
                })
                
            # Process RELATEDNAMES2 if available
            if 'RELATEDNAMES2' in row and pd.notna(row['RELATEDNAMES2']) and str(row['RELATEDNAMES2']).strip():
                processed_rows.append({
                    'LOINC_NUM': row['LOINC_NUM'], 
                    'LocaleCode': locale_code,
                    'LinguisticVariantName': row['RELATEDNAMES2'], 
                    'NameType': 'None'
                })
        
        print(f"  ✅ Processed {len(processed_rows)} linguistic variant entries from {filename}")
        return pd.DataFrame(processed_rows)
        
    except Exception as e:
        print(f"❌ Error processing {filename}: {e}")
        return pd.DataFrame()  # Return empty DataFrame on error

# Main linguistic variant processing
def process_all_linguistic_variants(linguistic_variants_path):
    """
    Process all valid linguistic variant files in the directory
    """
    print("🌍 Processing linguistic variant files...")
    
    # Get all CSV files in the directory
    all_csv_files = [
        os.path.join(linguistic_variants_path, f) 
        for f in os.listdir(linguistic_variants_path) 
        if f.endswith('.csv')
    ]
    
    # Filter to only proper linguistic variant files
    valid_linguistic_files = [
        filepath for filepath in all_csv_files 
        if is_proper_linguistic_variant_file(filepath)
    ]
    
    print(f"Found {len(all_csv_files)} CSV files, {len(valid_linguistic_files)} valid linguistic variant files")
    
    if not valid_linguistic_files:
        print("⚠️  No valid linguistic variant files found!")
        return pd.DataFrame()
    
    # Process each valid file
    all_processed_variants = []
    for file_path in valid_linguistic_files:
        processed_df = process_linguistic_file(file_path)
        if not processed_df.empty:
            all_processed_variants.append(processed_df)
    
    # Concatenate all results
    if all_processed_variants:
        final_df = pd.concat(all_processed_variants, ignore_index=True)
        print(f"🎉 Successfully processed {len(final_df)} total linguistic variant entries")
        return final_df
    else:
        print("⚠️  No linguistic variant data was successfully processed")
        return pd.DataFrame()

# Replace the existing linguistic variant processing code with this:
if mode in [0, 1]:
    # Process real linguistic variant files
    all_processed_variants = process_all_linguistic_variants(linguistic_variants_path)
else:
    # Use demo data for mode 2
    def load_csv_data(csv_data):
        return pd.read_csv(StringIO(csv_data))
    
    # Demo data processing (existing code can remain the same)
    esMX_df = load_csv_data(esMX_data)
    etEE_df = load_csv_data(etEE_data) 
    frBE_df = load_csv_data(frBE_data)
    
    processed_demo_data = []
    for df, locale in [(esMX_df, 'es-MX'), (etEE_df, 'et-EE'), (frBE_df, 'fr-BE')]:
        for _, row in df.iterrows():
            fsn = create_fsn(row)
            if fsn:
                processed_demo_data.append({
                    'LOINC_NUM': row['LOINC_NUM'],
                    'LocaleCode': locale,
                    'LinguisticVariantName': fsn,
                    'NameType': 'Fully-Specified'
                })
    
    all_processed_variants = pd.DataFrame(processed_demo_data)

# Continue with the existing pivot logic
if not all_processed_variants.empty:
    # Create a copy to avoid modifying the original dataframe
    temp_df = all_processed_variants.copy()
    
    # Sort the data to ensure the numeric counter is assigned consistently
    temp_df.sort_values(by=['LOINC_NUM', 'NameType', 'LocaleCode'], inplace=True)
    
    # Generate a numeric counter for each unique combination
    temp_df['counter'] = temp_df.groupby(['LOINC_NUM', 'NameType', 'LocaleCode']).cumcount() + 1
    
    # Create pivot column
    temp_df['pivot_column'] = ('names.' + temp_df['NameType'].astype(str) + '.' + 
                              temp_df['LocaleCode'].astype(str) + '[' + 
                              temp_df['counter'].astype(str) + ']')
    
    # Pivot the DataFrame
    pivoted_variants = temp_df.pivot(index='LOINC_NUM', 
                                   columns='pivot_column', 
                                   values='LinguisticVariantName')
    
    # Flatten and reset index
    pivoted_variants.columns = pivoted_variants.columns.get_level_values(0)
    pivoted_variants.reset_index(inplace=True)
    
    # Merge with the main LOINC DataFrame
    merged_loinc_df = pd.merge(loinc_df, pivoted_variants, on='LOINC_NUM', how='left')
    
    print(f"✅ Merged linguistic variants with {len(merged_loinc_df)} LOINC concepts")
else:
    # No linguistic variants to merge
    merged_loinc_df = loinc_df.copy()
    print("ℹ️  No linguistic variants to merge - using original LOINC data only")

🌍 Processing linguistic variant files...
Found 20 CSV files, 19 valid linguistic variant files
Processing deAT24LinguisticVariant.csv with locale code: de-AT
  ✅ Processed 7392 linguistic variant entries from deAT24LinguisticVariant.csv
Processing deDE15LinguisticVariant.csv with locale code: de-DE
  ✅ Processed 40749 linguistic variant entries from deDE15LinguisticVariant.csv
Processing elGR17LinguisticVariant.csv with locale code: el-GR
  ✅ Processed 179100 linguistic variant entries from elGR17LinguisticVariant.csv
Processing esAR7LinguisticVariant.csv with locale code: es-AR
  ✅ Processed 76586 linguistic variant entries from esAR7LinguisticVariant.csv
Processing esES12LinguisticVariant.csv with locale code: es-ES
  ✅ Processed 101068 linguistic variant entries from esES12LinguisticVariant.csv
Processing esMX28LinguisticVariant.csv with locale code: es-MX
  ✅ Processed 166462 linguistic variant entries from esMX28LinguisticVariant.csv
Processing etEE10LinguisticVariant.csv with loc

## Phase 2: Concept Creation

In this phase, we will transform LOINC terms, parts, answer lists, and linguistic variants into OCL Concept objects.

Phase 2 will receive the following from Phase 1:

    merged_loinc_df: A pandas DataFrame containing data from the Loinc.csv file, enriched with linguistic variants from all the linguistic variant CSV files. This DataFrame includes a wide range of LOINC fields and new columns for each linguistic variant and its locale.

    part_link_df: A pandas DataFrame loaded from the Part.csv file, containing information about LOINC parts, such as PartNumber, PartName, and PartTypeName.

    answer_list_df: A pandas DataFrame from the AnswerList.csv file, which includes details about LOINC answer lists and their corresponding answers.

### Expected Outputs of Phase 2

The output of Phase 2 will be a collection of OCL Concept objects, represented as a list of dictionaries. These objects will be generated from the dataframes provided by Phase 1 and will be structured to match the OCL format for Phase 5 ouputting. Specifically, this phase will produce the following dataframes:

    LOINC Concept Objects: Dictionaries that represent each LOINC code as an OCL concept, including its primary names, linguistic variants, and other attributes mapped to the extras field.

    LOINC Part Concept Objects: Dictionaries representing each unique LOINC part, with its PartNumber, name, and type properly mapped.

    LOINC Answer List Concept Objects: Dictionaries for each answer list, containing its ID and name.

    LOINC Answer Concept Objects: Dictionaries for each individual answer, linked to its parent answer list, including the display text and sequence number.

### Actions to be Taken

To produce the expected outputs, the code in Phase 2 will perform the following actions:

    Iterate and Map LOINC Data: It will loop through each row of the merged_loinc_df DataFrame. For each row, it will apply the loinc_to_ocl_mapping and fixed_values_loinc dictionaries to rename the LOINC fields. It will also apply the transformation_rules e.g. to set the retired status based on the STATUS field, along with other defined rules as needed. The new linguistic variant columns will be merged with the LOINC data to provide additional fields based on language.

    Create LOINC Part Concepts: The code will process the part_link_df DataFrame to create a distinct OCL Concept object for each unique PartNumber. It will use the loinc_part_to_ocl_mapping and fixed_values_loinc_parts to populate the OCL-specific fields.

    Generate Answer List and Answer Concepts: It will iterate through the answer_list_df DataFrame to create OCL Concept datasets, one each for LOINC Answers and Answer Lists. It will first create an object for each unique answer list (using answer_list_to_ocl_mapping and fixed_values_answer_list), and then create a separate OCL Concept object for each individual answer within those lists (using answer_to_ocl_mapping and fixed_values_answer). Duplicate rows will be deduplicated.

    Consolidate Concepts: All generated OCL Concept objects (for LOINCs, LOINC Parts, Answer Lists, and Answers) will be collected into a single, comprehensive list of dictionaries. This consolidated list will be the primary output for use in subsequent phases.

In [14]:
## Functions for creating OCL concepts for all LOINC types

def check_for_unmapped_fields(df, mapping, name_of_df, ignored_cols=None):
    """
    Checks for unmapped columns in a DataFrame and prints a warning message.
    """
    if ignored_cols is None:
        ignored_cols = []
    
    mapped_source_fields = set(mapping.keys())
    
    # Get all columns from the DataFrame that are not in the ignored list
    df_columns = set(df.columns) - set(ignored_cols)
    
    unmapped_fields = df_columns - mapped_source_fields
    
    if unmapped_fields:
        print(f"⚠️ Warning: Unmapped fields found in '{name_of_df}': {', '.join(sorted(list(unmapped_fields)))}")
        
def create_ocl_concept_from_row(row, mapping, fixed_values, transformation_rules=None):
    """
    Creates a single OCL Concept dictionary from a pandas DataFrame row,
    applying the specified mapping, fixed values, and transformation rules.
    This version keeps all attributes as top-level fields.
    """
    concept = fixed_values.copy()
    
    # Process column mappings
    for source_field, target_fields in mapping.items():
        if source_field in row and pd.notna(row[source_field]):
            if isinstance(target_fields, list):
                for target_field in target_fields:
                    concept[target_field] = row[source_field]
            else:
                concept[target_fields] = row[source_field]
    
    # Process transformation rules
    if transformation_rules:
        for rule in transformation_rules:
            fields_to_check = rule['field'] if isinstance(rule['field'], list) else [rule['field']]
            
            for field in fields_to_check:
                # Check for the existence of the field and handle both existing and missing values
                if field in row:
                    source_value = row[field]
                    
                    # Special handling for NaN, which cannot be a dictionary key
                    # Check if the value is NaN and if the rule has a transformation for NaN
                    if pd.isna(source_value):
                        if np.nan in rule['transformations']:
                            target_value = rule['transformations'][np.nan]
                            concept[rule['target_field']] = target_value
                            break # Stop after finding the first matching field and transforming
                    elif source_value in rule['transformations']:
                        target_value = rule['transformations'][source_value]
                        concept[rule['target_field']] = target_value
                        break # Stop after finding the first matching field and transforming
    return concept

def process_loinc_concepts(df):
    """
    Creates OCL Concept objects for all LOINC codes, including linguistic variants,
    with all attributes as top-level fields.
    """
    loinc_concepts = []
    
    linguistic_variant_cols = [col for col in df.columns if col.startswith('names.')]
    
    # Check for unmapped fields in the DataFrame
    ignored_cols = linguistic_variant_cols + ['PartNumber', 'ANSWERLISTID']
    check_for_unmapped_fields(df, loinc_to_ocl_mapping, 'merged_loinc_df', ignored_cols=ignored_cols)
    
    for _, row in df.iterrows():
        base_loinc_concept = create_ocl_concept_from_row(row, loinc_to_ocl_mapping, fixed_values_loinc, transformation_rules)

        # Handle dynamic linguistic variant names
        for col in linguistic_variant_cols:
            if pd.notna(row[col]):
                base_loinc_concept[col] = row[col]
        
        loinc_concepts.append(base_loinc_concept)
        
    return loinc_concepts

def process_loinc_parts(df):
    """
    Creates OCL Concept objects for unique LOINC Parts with all attributes
    as top-level fields.
    """
    loinc_part_concepts = []
    processed_part_numbers = set()
    
    # Check for unmapped fields
    check_for_unmapped_fields(df, loinc_part_to_ocl_mapping, 'part_link_df')

    for _, row in df.iterrows():
        part_number = row['PartNumber']
        if part_number not in processed_part_numbers:
            concept = create_ocl_concept_from_row(
                row, loinc_part_to_ocl_mapping, fixed_values_loinc_parts, transformation_rules
            )
            loinc_part_concepts.append(concept)
            processed_part_numbers.add(part_number)
            
    return loinc_part_concepts

def process_answer_lists_and_answers(df):
    """
    Creates OCL Concept objects for both LOINC Answer Lists and individual Answers,
    with all attributes as top-level fields.
    Deduplicates concepts to ensure uniqueness.
    """
    answer_list_concepts = []
    answer_concepts = []
    processed_answer_list_ids = set()
    processed_answer_ids = set()
    
    # Check for unmapped fields
    check_for_unmapped_fields(df, {**answer_list_to_ocl_mapping, **answer_to_ocl_mapping}, 'answer_list_df')

    for _, row in df.iterrows():
        # Create Answer List Concept
        answer_list_id = row['AnswerListId']
        if answer_list_id not in processed_answer_list_ids:
            answer_list_concept = create_ocl_concept_from_row(
                row, answer_list_to_ocl_mapping, fixed_values_answer_list, transformation_rules
            )
            answer_list_concepts.append(answer_list_concept)
            processed_answer_list_ids.add(answer_list_id)

        # Create Answer Concept
        answer_string_id = row['AnswerStringId']
        if answer_string_id not in processed_answer_ids:
            answer_concept = create_ocl_concept_from_row(
                row, answer_to_ocl_mapping, fixed_values_answer, transformation_rules
            )
            answer_concept['ParentAnswerListId'] = answer_list_id
            
            answer_concepts.append(answer_concept)
            processed_answer_ids.add(answer_string_id)
            
    return answer_list_concepts, answer_concepts

In [15]:
def phase_2_main(merged_loinc_df, part_link_df, answer_list_df):
    """
    Main function to execute all Phase 2 concept creation tasks and
    consolidate the outputs into a single list.
    """
    # Create LOINC Concepts
    print("Creating LOINC concepts...")
    loinc_concepts = process_loinc_concepts(merged_loinc_df)
    print(f"Created {len(loinc_concepts)} LOINC concepts from `merged_loinc_df`.")

    # Create LOINC Part Concepts
    print("Creating LOINC Part concepts...")
    loinc_part_concepts = process_loinc_parts(part_link_df)
    print(f"Created {len(loinc_part_concepts)} LOINC Part concepts from `part_link_df`.")

    # Create Answer List and Answer Concepts
    print("Creating Answer List and Answer concepts...")
    answer_list_concepts, answer_concepts = process_answer_lists_and_answers(answer_list_df)
    print(f"Created {len(answer_list_concepts)} Answer List concepts and {len(answer_concepts)} Answer concepts from `answer_list_df`.")
    
    # Consolidate all concepts into a single list
    all_concepts = loinc_concepts + loinc_part_concepts + answer_list_concepts + answer_concepts
    
    return all_concepts

# Execute the main function of Phase 2
all_ocl_concepts_list = phase_2_main(merged_loinc_df, part_link_df, answer_list_df)

# Convert the list of concepts into a pandas DataFrame.
all_ocl_concepts_df = pd.DataFrame(all_ocl_concepts_list)

# Now, sort the DataFrame's columns alphabetically.
all_ocl_concepts_df_sorted = all_ocl_concepts_df.sort_index(axis=1)

# Print the total number of concepts created.
print(f"\nTotal OCL Concepts created in Phase 2: {len(all_ocl_concepts_df_sorted)}")

# The sorted DataFrame is now stored in `all_ocl_concepts_df_sorted`.

Creating LOINC concepts...
Created 104672 LOINC concepts from `merged_loinc_df`.
Creating LOINC Part concepts...
Created 72740 LOINC Part concepts from `part_link_df`.
Creating Answer List and Answer concepts...
Created 4621 Answer List concepts and 20034 Answer concepts from `answer_list_df`.

Total OCL Concepts created in Phase 2: 202067


## Phase 3: Mapping Creation

This phase is for creating OCL Mapping objects based on relational files like `PanelsAndForms.csv` and `MapTo.csv`.

In [16]:
# --- DATA LOADING AND PREPARATION ---


try:
    df_panels_and_forms = pd.read_csv(panels_and_forms_csv_path, low_memory=False)
    df_loinc_answer_list_link = pd.read_csv(loinc_answer_list_link_csv_path, low_memory=False)
    df_answer_list = pd.read_csv(answer_list_csv_path, low_memory=False)
    df_loinc = pd.read_csv(loinc_csv_path, low_memory=False)
    df_map_to = pd.read_csv(map_to_csv_path, low_memory=False)
    print("All DataFrames loaded successfully.")
    
except NameError:
    # Fallback to hardcoded file names if path variables are not defined.
    print("Warning: Path variables not found. Attempting to load from hardcoded filenames.")
    try:
        df_panels_and_forms = pd.read_csv("PanelsAndForms.csv")
        df_loinc_answer_list_link = pd.read_csv("LoincAnswerListLink.csv")
        df_answer_list = pd.read_csv("AnswerList.csv")
        df_loinc = pd.read_csv("Loinc.csv")
        df_map_to = pd.read_csv("MapTo.csv")
        print("DataFrames loaded from hardcoded filenames.")
    except FileNotFoundError as e:
        print(f"Error: One or more files not found: {e}")
        # Initialize empty DataFrames to prevent further errors
        df_panels_and_forms = pd.DataFrame()
        df_loinc_answer_list_link = pd.DataFrame()
        df_answer_list = pd.DataFrame()
        df_loinc = pd.DataFrame()
        df_map_to = pd.DataFrame()

except FileNotFoundError as e:
    print(f"Error: One or more files not found: {e}")
    # Initialize empty DataFrames to prevent further errors
    df_panels_and_forms = pd.DataFrame()
    df_loinc_answer_list_link = pd.DataFrame()
    df_answer_list = pd.DataFrame()
    df_loinc = pd.DataFrame()
    df_map_to = pd.DataFrame()

All DataFrames loaded successfully.


In [17]:
# Process the mappings.
def process_mappings(df, config):
    """
    Generates OCL mapping objects from a DataFrame using a configuration dictionary.
    """
    mapping_objects = []
    
    # Process each row of the input DataFrame
    for _, row in df.iterrows():
        ocl_object = {}
        
        # Populate fixed and mapped fields
        for field_map in config["field_mappings"]:
            ocl_field = field_map["ocl_field"]
            
            # Handle fixed values
            if "source_value" in field_map:
                value = field_map["source_value"]
            # Handle direct field mappings with an optional rule
            elif "source_field" in field_map:
                source_field = field_map["source_field"]
                value = row.get(source_field)
                if "rule" in field_map:
                    if pd.notna(value):
                        value = field_map["rule"](value)
                    else:
                        # NEW: Assign None if the source value is not a number and no rule is applied
                        value = None
            # Handle combined field mappings (e.g., for Cardinality)
            elif "source_field_min" in field_map and "source_field_max" in field_map:
                min_val = row.get(field_map["source_field_min"])
                max_val = row.get(field_map["source_field_max"])
                if pd.notna(min_val) and pd.notna(max_val):
                    value = field_map["rule"](min_val, max_val)
                else:
                    value = None
            else:
                continue

            # Populate the OCL object, handling the 'extras' dictionary
            if ocl_field.startswith('extras'):
                if "extras" not in ocl_object:
                    ocl_object["extras"] = {}
                key = ocl_field.split('"')[1]
                ocl_object["extras"][key] = value
            else:
                ocl_object[ocl_field] = value
        
        # Only append valid OCL objects if both `from_concept_url` and `to_concept_url` are present.
        if ocl_object.get("from_concept_url") and ocl_object.get("to_concept_url"):
            mapping_objects.append(ocl_object)
            
    return mapping_objects

# --- MAPPING EXECUTION ---

# Assuming your DataFrames are already loaded (e.g., df_loinc, df_panels_and_forms, etc.)

# Initialize mapping lists to empty
panel_to_test_mappings = []
Youtube_mappings = []
order_entry_mappings = []
code_evolution_mappings = []
associated_observations_mappings = []

# Process the Panel-to-Test mappings.
panel_to_test_mappings = process_mappings(df_panels_and_forms, MAPPING_CONFIGS["Panel-to-Test"])
print(f"Generated {len(panel_to_test_mappings)} Panel-to-Test mappings.")
print("-" * 20)

# Process the Question-to-Answer mappings.
join_info = MAPPING_CONFIGS["Question-to-Answer"]["join_condition"]
left_col = join_info["left_on"].split('.')[-1]
right_col = join_info["right_on"].split('.')[-1]
df_joined_qa = pd.merge(df_loinc_answer_list_link, df_answer_list, left_on=left_col, right_on=right_col, suffixes=("_LoincAnswerListLink", "_AnswerList"))
Youtube_mappings = process_mappings(df_joined_qa, MAPPING_CONFIGS["Question-to-Answer"])
print(f"Generated {len(Youtube_mappings)} Question-to-Answer mappings.")
print("-" * 20)

# Process the Ask at Order Entry mappings.
order_entry_mappings = process_mappings(df_loinc, MAPPING_CONFIGS["Ask at Order Entry"])
print(f"Generated {len(order_entry_mappings)} Ask at Order Entry mappings.")
print("-" * 20)

# Process the Code Evolution mappings.
code_evolution_mappings = process_mappings(df_map_to, MAPPING_CONFIGS["Code Evolution"])
print(f"Generated {len(code_evolution_mappings)} Code Evolution mappings.")
print("-" * 20)

# Process the Associated Observations mappings.
def preprocess_associated_observations(df):
    """
    Preprocesses the LOINC DataFrame to expand rows with delimited AssociatedObservations values.
    Each semicolon-separated LOINC code becomes its own row for mapping creation.
    
    Args:
        df: DataFrame containing LOINC data with AssociatedObservations column
        
    Returns:
        Expanded DataFrame with individual rows for each associated observation
    """
    import re
    
    def is_valid_loinc_format(code):
        """Check if a string matches basic LOINC format (NNNNN-N)"""
        if not code or pd.isna(code):
            return False
        code = str(code).strip()
        # LOINC pattern: 1-5 digits, hyphen, 1-2 digits
        return bool(re.match(r'^\d{1,5}-\d{1,2}$', code))
    
    def extract_loinc_codes(associated_obs_value):
        """Extract valid LOINC codes from AssociatedObservations field"""
        if pd.isna(associated_obs_value):
            return []
        
        # Convert to string and split on common delimiters
        text = str(associated_obs_value).strip()
        if not text:
            return []
        
        # Split on semicolons, commas, or pipe characters
        potential_codes = re.split(r'[;,|]', text)
        
        valid_codes = []
        for code in potential_codes:
            code = code.strip()
            if is_valid_loinc_format(code):
                valid_codes.append(code)
        
        return valid_codes
    
    # Filter to only rows with AssociatedObservations values
    has_assoc_obs = df['AssociatedObservations'].notna() & (df['AssociatedObservations'].astype(str).str.strip() != '')
    df_with_assoc = df[has_assoc_obs].copy()
    
    if df_with_assoc.empty:
        print("No rows found with AssociatedObservations values")
        return pd.DataFrame()
    
    print(f"Found {len(df_with_assoc)} rows with AssociatedObservations values")
    
    # Expand rows with multiple associated observations
    expanded_rows = []
    delimited_count = 0
    total_mappings = 0
    
    for _, row in df_with_assoc.iterrows():
        associated_codes = extract_loinc_codes(row['AssociatedObservations'])
        
        if len(associated_codes) > 1:
            delimited_count += 1
            
        for code in associated_codes:
            # Create a new row for each associated observation
            new_row = row.copy()
            new_row['AssociatedObservations'] = code
            expanded_rows.append(new_row)
            total_mappings += 1
    
    if not expanded_rows:
        print("No valid LOINC codes found in AssociatedObservations fields")
        return pd.DataFrame()
    
    expanded_df = pd.DataFrame(expanded_rows)
    
    print(f"Expanded {delimited_count} rows with delimited values")
    print(f"Created {total_mappings} individual mapping rows")
    # print(f"Sample expanded AssociatedObservations values: {list(expanded_df['AssociatedObservations'].head(10))}")
    
    return expanded_df


print("=== Processing Associated Observations Mappings ===")

# Expand the LOINC dataframe for Associated Observations
df_loinc_expanded_assoc = preprocess_associated_observations(df_loinc)

if not df_loinc_expanded_assoc.empty:
    # Process the expanded dataframe through the normal mapping process
    associated_observations_mappings = process_mappings(df_loinc_expanded_assoc, MAPPING_CONFIGS["Associated Observations"])
    print(f"Generated {len(associated_observations_mappings)} Associated Observations mappings.")
else:
    # Fallback to original processing if no expansion occurred
    print("No delimited values found, using original processing...")
    associated_observations_mappings = process_mappings(df_loinc, MAPPING_CONFIGS["Associated Observations"])
    print(f"Generated {len(associated_observations_mappings)} Associated Observations mappings.")

print("-" * 20)

# Combine all mappings into a single list for the final output
all_mappings = (
    panel_to_test_mappings +
    Youtube_mappings +
    order_entry_mappings +
    code_evolution_mappings +
    associated_observations_mappings
)
print(f"\nTotal OCL mapping objects created for Round 1: {len(all_mappings)}")

Generated 91993 Panel-to-Test mappings.
--------------------
Generated 145765 Question-to-Answer mappings.
--------------------
Generated 65 Ask at Order Entry mappings.
--------------------
Generated 4643 Code Evolution mappings.
--------------------
=== Processing Associated Observations Mappings ===
Found 8646 rows with AssociatedObservations values
Expanded 8006 rows with delimited values
Created 17466 individual mapping rows
Generated 17466 Associated Observations mappings.
--------------------

Total OCL mapping objects created for Round 1: 259932


## Phase 4: Hierarchy Creation

Phase 4 focuses on parsing the `ComponentHierarchyBySystem.csv` file and establishing hierarchy relationships between concepts. One single hierarchy will be created in the end, which means that every concept will have a field 'parent_concept_urls' with one or more concept URLs (i.e. the URL prefix plus the ID of the parent concept). One example: "parent_concept_urls":["/orgs/Regenstrief/sources/LOINC-3/concepts/ROOT/"]

The Hierarchy includes:
* Component-by-System Hierarchy - Connects LOINC Parts or LOINCs to a parent part
* Top-level Containers - Connects parts and terms without parents to a Container concept
* List-to-Answer Hierarchy - Links LOINC Answers to their Answer List parent(s)

This phase creates a set of Container concepts, whose parent concept is the special "ROOT" concept. ROOT is the only concept without a parent. This Phase should assign parent concept URLs, along with identifying and listing concepts (grouped by their concept_class) that do not yet have a parent.

In [18]:
### Hierarchy Analysis Functions ###

def identify_node_types(df):
    """
    Identifies LOINC Parts (branches) and LOINC terms (leaf nodes) in the hierarchy.
    LOINC Parts are the branches, and LOINC terms are the leaf nodes.
    """
    # LOINC Parts (branches) appear in the 'parent_concept' column.
    loinc_parts = df['parent_concept'].dropna().unique()
    # LOINC terms (leaf nodes) are 'match-id' values that are not 'parent_concept' values.
    loinc_terms = df[~df['match-id'].isin(loinc_parts)]['match-id'].unique()
    
    print("\nLOINC Hierarchy Breakdown:")
    print(f"Total unique codes: {df['match-id'].nunique()}")
    print(f"Number of LOINC Parts (branches): {len(loinc_parts)}")
    print(f"Number of LOINC Terms (leaf nodes): {len(loinc_terms)}")
    return loinc_parts, loinc_terms

# Execute the analysis functions if data was loaded successfully.
if hierarchy_df is not None:
    loinc_parts, loinc_terms = identify_node_types(hierarchy_df)
else:
    print("Error: The hierarchy_df is not available. Please check the data loading step.")


LOINC Hierarchy Breakdown:
Total unique codes: 176668
Number of LOINC Parts (branches): 55871
Number of LOINC Terms (leaf nodes): 120797


In [19]:
# Merge and parent URL creation

print("🔧 Implementing Multi-Parent Hierarchy Support")
print("=" * 60)

# Step 1: Don't deduplicate hierarchy_df - we want to preserve all relationships
print(f"📊 Preserving all hierarchy relationships:")
print(f"   Total hierarchy records: {len(hierarchy_df):,}")
print(f"   Unique concepts: {hierarchy_df['match-id'].nunique():,}")
print(f"   Concepts with multiple parents: {len(hierarchy_df) - hierarchy_df['match-id'].nunique():,}")

# Step 2: Merge all concepts with all hierarchy records (this will create duplicates)
print(f"\n🔗 Merging concepts with hierarchy data...")
concepts_with_hierarchy_expanded = pd.merge(
    all_ocl_concepts_df_sorted, 
    hierarchy_df, 
    left_on='id', 
    right_on='match-id', 
    how='left'
)
print(f"   After merge: {len(concepts_with_hierarchy_expanded):,} rows (includes duplicates)")

# Step 3: Group by concept ID and aggregate parent concepts
print(f"\n🔄 Consolidating multiple parents per concept...")

def combine_parent_concepts(group):
    """Combine multiple parent concepts for a single concept."""
    # Get the first row as the base (all non-hierarchy fields should be identical)
    base_row = group.iloc[0].copy()
    
    # Collect all unique parent concepts for this concept
    parent_concepts = []
    hierarchy_paths = []
    hierarchy_sequences = []
    
    for _, row in group.iterrows():
        if pd.notna(row.get('parent_concept')):
            parent_concepts.append(str(row['parent_concept']))  # Convert to string to ensure consistency
        if pd.notna(row.get('extras.PATH_TO_ROOT')):
            hierarchy_paths.append(str(row['extras.PATH_TO_ROOT']))
        if pd.notna(row.get('extras.HIERARCHY_SEQUENCE')):
            hierarchy_sequences.append(row['extras.HIERARCHY_SEQUENCE'])
    
    # Remove duplicates while preserving order - ensure we get Python lists
    unique_parents = list(dict.fromkeys(parent_concepts))  # Preserves order, removes duplicates
    unique_paths = list(dict.fromkeys(hierarchy_paths))
    unique_sequences = list(dict.fromkeys(hierarchy_sequences))
    
    # Store multiple parents and hierarchy info - ensure Python objects, not pandas objects
    if unique_parents:
        if len(unique_parents) > 1:
            base_row['parent_concept'] = unique_parents  # Python list
        else:
            base_row['parent_concept'] = unique_parents[0]  # Single string
    else:
        base_row['parent_concept'] = None
        
    if unique_paths:
        if len(unique_paths) > 1:
            base_row['extras.PATH_TO_ROOT'] = unique_paths  # Python list
        else:
            base_row['extras.PATH_TO_ROOT'] = unique_paths[0]  # Single string
    else:
        base_row['extras.PATH_TO_ROOT'] = None
    
    if unique_sequences:
        if len(unique_sequences) > 1:
            base_row['extras.HIERARCHY_SEQUENCE'] = unique_sequences  # Python list
        else:
            base_row['extras.HIERARCHY_SEQUENCE'] = unique_sequences[0]  # Single value
    else:
        base_row['extras.HIERARCHY_SEQUENCE'] = None
    
    return base_row

# Apply the grouping and consolidation
concepts_with_hierarchy_df = concepts_with_hierarchy_expanded.groupby('id').apply(combine_parent_concepts).reset_index(drop=True)

print(f"   After consolidation: {len(concepts_with_hierarchy_df):,} rows (one per concept)")

# Step 4: Create parent_concept_urls supporting multiple parents
def create_parent_urls_multi(row):
    """
    Creates a list of parent concept URLs for a given row, supporting multiple parents.
    FIXED: Properly handles NaN values to prevent "nan" string parents.
    """
    parent_urls = []
    
    # Handle hierarchy parent(s)
    parent_concept = row.get('parent_concept')
    
    if parent_concept is not None:
        if isinstance(parent_concept, list):
            # Multiple hierarchy parents
            for parent in parent_concept:
                if (parent is not None and 
                    not pd.isna(parent) and 
                    str(parent).strip() and 
                    str(parent).lower() != 'nan'):  # Exclude "nan" strings
                    parent_urls.append(CONCEPT_URL_PREFIX.format(str(parent)))
        else:
            # Single hierarchy parent
            if (parent_concept is not None and 
                not pd.isna(parent_concept) and 
                str(parent_concept).strip() and 
                str(parent_concept).lower() != 'nan'):  # Exclude "nan" strings
                parent_urls.append(CONCEPT_URL_PREFIX.format(str(parent_concept)))
    
    # Handle Answer List parent (from Phase 2) - FIXED NaN handling
    answer_list_parent = row.get('ParentAnswerListId')
    if (answer_list_parent is not None and 
        not pd.isna(answer_list_parent) and 
        str(answer_list_parent).strip() and 
        str(answer_list_parent).lower() != 'nan'):  # Exclude "nan" strings
        parent_urls.append(CONCEPT_URL_PREFIX.format(str(answer_list_parent)))
    
    # Return the list of parent URLs, or None if empty
    return parent_urls if parent_urls else None

# Apply the multi-parent URL creation
concepts_with_hierarchy_df['parent_concept_urls'] = concepts_with_hierarchy_df.apply(create_parent_urls_multi, axis=1)

# Step 5: Report the results
print(f"\n📊 Multi-Parent Hierarchy Results:")
print(f"   Total concepts: {len(concepts_with_hierarchy_df):,}")

# Count concepts by number of parents
parent_counts = concepts_with_hierarchy_df['parent_concept_urls'].apply(
    lambda x: len(x) if isinstance(x, list) else (1 if x is not None else 0)
).value_counts().sort_index()

print(f"   Concepts by number of parents:")
for num_parents, count in parent_counts.items():
    if num_parents == 0:
        print(f"     No parents: {count:,} concepts")
    elif num_parents == 1:
        print(f"     Single parent: {count:,} concepts")
    else:
        print(f"     {num_parents} parents: {count:,} concepts")

# Show examples of concepts with multiple parents
multi_parent_concepts = concepts_with_hierarchy_df[
    concepts_with_hierarchy_df['parent_concept_urls'].apply(
        lambda x: isinstance(x, list) and len(x) > 1
    )
]

if not multi_parent_concepts.empty:
    print(f"\n🔍 Examples of concepts with multiple parents:")
    for i, (_, row) in enumerate(multi_parent_concepts.head(5).iterrows()):
        concept_id = row['id']
        parents = row['parent_concept_urls']
        parent_ids = [url.split('/')[-2] for url in parents]  # Extract parent IDs from URLs
        print(f"     {concept_id}: {len(parents)} parents -> {', '.join(parent_ids)}")

print(f"\n✅ Multi-parent hierarchy implementation complete!")
print("=" * 60)

# Deduplicate to ensure no duplicate concepts (should be unnecessary but safety check)
concepts_with_hierarchy_df = concepts_with_hierarchy_df.drop_duplicates(subset=['id'], keep='first')

print(f"Final concept count: {len(concepts_with_hierarchy_df):,}")

🔧 Implementing Multi-Parent Hierarchy Support
📊 Preserving all hierarchy relationships:
   Total hierarchy records: 180,607
   Unique concepts: 176,668
   Concepts with multiple parents: 3,939

🔗 Merging concepts with hierarchy data...
   After merge: 205,078 rows (includes duplicates)

🔄 Consolidating multiple parents per concept...


C:\Users\jamlung\AppData\Local\Temp\1\ipykernel_32676\36866622.py:77: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  concepts_with_hierarchy_df = concepts_with_hierarchy_expanded.groupby('id').apply(combine_parent_concepts).reset_index(drop=True)


   After consolidation: 202,066 rows (one per concept)

📊 Multi-Parent Hierarchy Results:
   Total concepts: 202,066
   Concepts by number of parents:
     No parents: 46,849 concepts
     Single parent: 154,596 concepts
     2 parents: 551 concepts
     3 parents: 53 concepts
     4 parents: 13 concepts
     5 parents: 3 concepts
     6 parents: 1 concepts

🔍 Examples of concepts with multiple parents:
     LP100617-2: 2 parents -> LP31465-5, LP7840-4
     LP101975-3: 3 parents -> LP7779-4, LP7782-8, LP7778-6
     LP101977-9: 2 parents -> LP29672-0, LP7778-6
     LP101982-9: 2 parents -> LP172861-9, LP7840-4
     LP113911-4: 2 parents -> LP31406-9, LP111174-1

✅ Multi-parent hierarchy implementation complete!
Final concept count: 202,066


## Phase 5: UMLS Enhancement

Here, we'll query an external UMLS API to enrich the concepts with CUIs (Concept Unique Identifiers).

In [20]:
# # Analysis and Debug of LOINC Hierarchy - Optional

# print("🔍 DEBUGGING: Checking for duplicates in concepts_with_hierarchy_df")
# print("=" * 60)

# # Check the input dataframe for duplicates
# print(f"📊 concepts_with_hierarchy_df shape: {concepts_with_hierarchy_df.shape}")
# print(f"   Total rows: {len(concepts_with_hierarchy_df):,}")
# print(f"   Unique IDs: {concepts_with_hierarchy_df['id'].nunique():,}")

# # Check for duplicate IDs
# duplicate_ids = concepts_with_hierarchy_df[concepts_with_hierarchy_df['id'].duplicated(keep=False)]
# if not duplicate_ids.empty:
#     print(f"❌ FOUND {len(duplicate_ids):,} rows with duplicate IDs!")
#     print(f"   Number of unique duplicate IDs: {duplicate_ids['id'].nunique()}")
    
#     # Show examples of duplicates
#     print("\n🔍 Examples of duplicate IDs:")
#     sample_duplicates = duplicate_ids.groupby('id').size().head(10)
#     for concept_id, count in sample_duplicates.items():
#         print(f"   {concept_id}: {count} copies")
        
#     # Show details of first duplicate
#     first_duplicate_id = sample_duplicates.index[0]
#     duplicate_rows = concepts_with_hierarchy_df[concepts_with_hierarchy_df['id'] == first_duplicate_id]
#     print(f"\n📋 Details of duplicate ID '{first_duplicate_id}':")
#     print(f"   Columns that differ between duplicates:")
    
#     # Check which columns differ between the duplicate rows
#     for col in duplicate_rows.columns:
#         unique_values = duplicate_rows[col].nunique()
#         if unique_values > 1:
#             print(f"     {col}: {unique_values} different values")
#             print(f"       Values: {list(duplicate_rows[col].unique())}")
        
# else:
#     print("✅ No duplicate IDs found in concepts_with_hierarchy_df")

# # Check if the issue might be in hierarchy merge
# print("\n🔍 Checking hierarchy_df for issues...")
# if 'hierarchy_df' in globals():
#     print(f"📊 hierarchy_df shape: {hierarchy_df.shape}")
#     if 'match-id' in hierarchy_df.columns:
#         print(f"   Unique match-ids: {hierarchy_df['match-id'].nunique():,}")
#         duplicate_match_ids = hierarchy_df[hierarchy_df['match-id'].duplicated(keep=False)]
#         if not duplicate_match_ids.empty:
#             print(f"❌ hierarchy_df has {len(duplicate_match_ids):,} rows with duplicate match-ids!")
#             sample_hierarchy_dups = duplicate_match_ids['match-id'].value_counts().head(5)
#             print("   Examples:")
#             for match_id, count in sample_hierarchy_dups.items():
#                 print(f"     {match_id}: {count} copies")
#         else:
#             print("✅ No duplicate match-ids in hierarchy_df")

# # Check all_ocl_concepts_df_sorted (the original concepts)
# print("\n🔍 Checking all_ocl_concepts_df_sorted for issues...")
# if 'all_ocl_concepts_df_sorted' in globals():
#     print(f"📊 all_ocl_concepts_df_sorted shape: {all_ocl_concepts_df_sorted.shape}")
#     print(f"   Unique IDs: {all_ocl_concepts_df_sorted['id'].nunique():,}")
    
#     original_duplicates = all_ocl_concepts_df_sorted[all_ocl_concepts_df_sorted['id'].duplicated(keep=False)]
#     if not original_duplicates.empty:
#         print(f"❌ ORIGINAL concepts dataframe has {len(original_duplicates):,} duplicate IDs!")
#         print("   This is the root cause of the problem!")
#     else:
#         print("✅ Original concepts dataframe has no duplicates")

# print("=" * 60)
# print("🎯 DIAGNOSIS COMPLETE - Check the output above to identify the source")

In [22]:
# This phase uses the enhanced SimpleLOINCMapper to map LOINC codes to UMLS CUIs.
# The enhanced version includes:
# - Multi-level caching (Local UMLS cache + API results cache)
# - Config file support for centralized settings
# - Bulk processing capabilities for improved performance
# - Automatic cache discovery and intelligent fallback
# - Comprehensive error handling and retry logic
# - Progress reporting and performance statistics

print("🚀 Starting Enhanced LOINC to UMLS CUI Mapping...")
print("=" * 60)

try:
    # Import the enhanced SimpleLOINCMapper
    from simple_loinc_mapper import SimpleLOINCMapper
    print("✅ Successfully imported SimpleLOINCMapper")
except ImportError as e:
    print(f"❌ Error importing SimpleLOINCMapper: {e}")
    print("   Please ensure simple_loinc_mapper.py is in the current directory")
    print("   You can find the enhanced version in the project knowledge")
    raise

try:
    # Initialize the enhanced mapper with config file support
    # This automatically discovers UMLS cache files and loads API settings
    mapper = SimpleLOINCMapper(config_path="UMLS_API_config.json")
    print("✅ Successfully initialized enhanced SimpleLOINCMapper")
    
    # Display cache information
    cache_info = mapper.get_cache_info()
    print(f"📊 Cache Status:")
    print(f"   Local UMLS cache: {cache_info['umls_cache_size']:,} mappings")
    print(f"   API results cache: {cache_info['api_cache_size']:,} mappings") 
    print(f"   Total cached mappings: {cache_info['total_cache_size']:,}")
    
except Exception as e:
    print(f"❌ Error initializing mapper: {e}")
    print("   Falling back to basic initialization...")
    try:
        # Fallback: Load config manually for backward compatibility
        with open('UMLS_API_config.json', 'r') as f:
            config = json.load(f)
            api_key = config.get('api_key')
            rate_limit_delay = config.get('rate_limit_delay', 0.2)

        if not api_key or api_key == "YOUR_UMLS_API_KEY_HERE":
            print("⚠️  Warning: No valid UMLS API key found in config file")
            print("   Only cached results will be available")
            mapper = SimpleLOINCMapper()
        else:
            mapper = SimpleLOINCMapper(api_key=api_key, rate_limit=rate_limit_delay)
            print("✅ Initialized with basic configuration")
    except Exception as fallback_error:
        print(f"❌ Fallback initialization failed: {fallback_error}")
        raise

# Filter concepts to exclude Answer Lists (they don't typically have UMLS mappings)
print("\n📋 Preparing LOINC codes for mapping...")
loinc_codes_all = concepts_with_hierarchy_df['id'].unique()
filtered_concepts_df = concepts_with_hierarchy_df[concepts_with_hierarchy_df['concept_class'] != 'Answer List']
loinc_codes_to_map = filtered_concepts_df['id'].unique()

print(f"   Total unique concept IDs: {len(loinc_codes_all):,}")
print(f"   LOINC codes to map (excluding Answer Lists): {len(loinc_codes_to_map):,}")
print(f"   Answer Lists excluded from mapping: {len(loinc_codes_all) - len(loinc_codes_to_map):,}")

# Estimate performance based on cache hit rate
if hasattr(mapper, 'umls_cache') and mapper.umls_cache:
    cached_codes = sum(1 for code in loinc_codes_to_map if code in mapper.umls_cache)
    cache_hit_rate = (cached_codes / len(loinc_codes_to_map)) * 100 if loinc_codes_to_map.size > 0 else 0
    estimated_api_calls = len(loinc_codes_to_map) - cached_codes
    estimated_time_minutes = (estimated_api_calls * 0.2) / 60  # 0.2 seconds per API call
    
    print(f"📈 Performance Estimate:")
    print(f"   Cache hit rate: {cache_hit_rate:.1f}% ({cached_codes:,} codes)")
    print(f"   Estimated API calls needed: {estimated_api_calls:,}")
    print(f"   Estimated processing time: {estimated_time_minutes:.1f} minutes")

# Process LOINC codes using enhanced bulk processing
print(f"\n📄 Processing {len(loinc_codes_to_map):,} LOINC codes...")
print("   Using enhanced bulk processing with intelligent caching...")

start_time = time.time()

# Use the enhanced bulk processing method
# This method automatically handles caching, retries, and progress reporting
successful_mappings, failed_codes = mapper.process_loinc_list(
    loinc_codes_to_map.tolist(), 
    progress_interval=5000,  # Report progress every 5000 codes
    force_refresh=False  # Use existing caches
)

end_time = time.time()
processing_time = end_time - start_time

# Display comprehensive results
print(f"\n📊 Mapping Results Summary:")
print(f"   Total codes processed: {len(loinc_codes_to_map):,}")
print(f"   Successful mappings: {len(successful_mappings):,}")
print(f"   Failed mappings: {len(failed_codes):,}")
print(f"   Success rate: {(len(successful_mappings) / len(loinc_codes_to_map) * 100):.1f}%")
print(f"   Processing time: {processing_time:.1f} seconds ({processing_time/60:.1f} minutes)")

if len(successful_mappings) > 0:
    avg_time_per_code = processing_time / len(loinc_codes_to_map)
    print(f"   Average time per code: {avg_time_per_code*1000:.2f} milliseconds")

# Display performance statistics if available
if hasattr(mapper, 'stats'):
    stats = mapper.stats
    print(f"\n📈 Detailed Performance Statistics:")
    print(f"   UMLS cache hits: {stats.get('umls_cache_hits', 0):,}")
    print(f"   API cache hits: {stats.get('api_cache_hits', 0):,}")
    print(f"   New API calls: {stats.get('api_calls', 0):,}")
    print(f"   Not found: {stats.get('not_found', 0):,}")

# Create DataFrame from successful mappings
if successful_mappings:
    # print(f"\n📋 Creating mapping DataFrame...")
    
    # DEBUG: Inspect the actual structure of successful mappings
    # print(f"🔍 Debug: Inspecting mapping structure...")
    # if len(successful_mappings) > 0:
    #     sample_mapping = successful_mappings[0]
    #     print(f"   Sample mapping keys: {list(sample_mapping.keys())}")
    #     print(f"   Sample mapping: {sample_mapping}")
    
    # Try to create DataFrame
    try:
        loinc_cui_df = pd.DataFrame(successful_mappings)
        print(f"   Successfully created DataFrame with {len(loinc_cui_df)} rows")
        if len(loinc_cui_df.columns) > 0:
            print(f"   DataFrame columns: {list(loinc_cui_df.columns)}")
    except Exception as e:
        print(f"   ❌ Error creating DataFrame: {e}")
        loinc_cui_df = pd.DataFrame()
    
    # Display sample mappings with error handling
    print(f"✅ Sample successful mappings:")
    # for i, mapping in enumerate(successful_mappings[:3]):
    #     try:
    #         # Try the expected keys first
    #         if 'loinc_code' in mapping and 'cui' in mapping:
    #             method = mapping.get('mapping_method', 'unknown')
    #             print(f"   {mapping['loinc_code']} → {mapping['cui']} (via {method})")
    #         else:
    #             # If expected keys don't exist, show what we have
    #             print(f"   Mapping {i+1}: {mapping}")
    #     except Exception as e:
    #         print(f"   ❌ Error displaying mapping {i+1}: {e}")
    #         print(f"   Raw mapping: {mapping}")
        
    #     if i >= 2:  # Show max 3 examples
    #         break
    
    # if len(successful_mappings) > 3:
    #     print(f"   ... and {len(successful_mappings) - 3:,} more mappings")
else:
    print("⚠️  No successful mappings found - creating empty DataFrame")
    loinc_cui_df = pd.DataFrame()

# Merge results with the main concepts DataFrame
print(f"\n📄 Merging UMLS mappings with concept data...")

if not loinc_cui_df.empty and 'loinc_code' in loinc_cui_df.columns and 'cui' in loinc_cui_df.columns:
    # Prepare the mapping data for merge
    loinc_cui_for_merge = loinc_cui_df[['loinc_code', 'cui']].copy()
    
    # Merge with the main concepts DataFrame
    merged_loinc_cui_df = pd.merge(
        concepts_with_hierarchy_df, 
        loinc_cui_for_merge, 
        left_on='id', 
        right_on='loinc_code', 
        how='left'
    )
    
    # Clean up merge artifacts
    if 'loinc_code' in merged_loinc_cui_df.columns:
        merged_loinc_cui_df.drop(columns=['loinc_code'], inplace=True)
    
    # Rename and add UMLS fields
    merged_loinc_cui_df.rename(columns={'cui': 'external_id'}, inplace=True)
    merged_loinc_cui_df['extras.UMLS_CUI'] = merged_loinc_cui_df['external_id']
    
    # Count results
    matched_cui_count = merged_loinc_cui_df['external_id'].notna().sum()
    unmatched_cui_count = merged_loinc_cui_df['external_id'].isna().sum()
    
    print(f"✅ Merge completed successfully:")
    print(f"   Total concepts in final dataset: {len(merged_loinc_cui_df):,}")
    print(f"   Concepts with UMLS CUI: {matched_cui_count:,}")
    print(f"   Concepts without UMLS CUI: {unmatched_cui_count:,}")
    print(f"   UMLS mapping coverage: {(matched_cui_count / len(merged_loinc_cui_df) * 100):.1f}%")

else:
    print("⚠️  No mappings to merge or missing expected columns - using original concepts DataFrame")
    merged_loinc_cui_df = concepts_with_hierarchy_df.copy()
    matched_cui_count = 0
    unmatched_cui_count = len(merged_loinc_cui_df)

# Display failed codes summary if any
if failed_codes:
    print(f"\n⚠️  Failed Mappings Summary:")
    print(f"   Total failed codes: {len(failed_codes):,}")
    print(f"   Sample failed codes: {failed_codes[:5]}")
    if len(failed_codes) > 5:
        print(f"   ... and {len(failed_codes) - 5:,} more")
    
    # Optionally save failed codes for analysis
    failed_codes_file = os.path.join(output_folder, 'failed_umls_mappings.txt')
    with open(failed_codes_file, 'w') as f:
        for code in failed_codes:
            f.write(f"{code}\n")
    print(f"   Failed codes saved to: {failed_codes_file}")

print(f"\n✅ Phase 5 UMLS Enhancement Complete!")
print(f"   Enhanced SimpleLOINCMapper delivered significant performance improvements")
print(f"   through intelligent multi-level caching and bulk processing.")
print(f"   Proceed to Phase 6 for final output generation.")
print(f"=" * 60)

🚀 Starting Enhanced LOINC to UMLS CUI Mapping...
✅ Successfully imported SimpleLOINCMapper
2025-08-19 10:00:36,659 - INFO - Loaded configuration from UMLS_API_config.json
2025-08-19 10:00:36,790 - INFO - Loaded 245,133 LOINC mappings from local UMLS cache: output/loinc_cui_simple_lookup.json
2025-08-19 10:00:36,791 - INFO - No existing API cache file found. Starting with an empty cache.
2025-08-19 10:00:36,793 - INFO - Initialized with local UMLS cache: 245,133 mappings
2025-08-19 10:00:36,795 - INFO - Initialized with API cache: 0 mappings
✅ Successfully initialized enhanced SimpleLOINCMapper
📊 Cache Status:
   Local UMLS cache: 245,133 mappings
   API results cache: 0 mappings
   Total cached mappings: 245,133

📋 Preparing LOINC codes for mapping...
   Total unique concept IDs: 202,066
   LOINC codes to map (excluding Answer Lists): 197,445
   Answer Lists excluded from mapping: 4,621
📈 Performance Estimate:
   Cache hit rate: 99.5% (196,436 codes)
   Estimated API calls needed: 1,00

## Phase 6: Output Generation

The final phase is to save all the created OCL objects to a JSON files (.json) in the Bulk Import JSON-lines-like format for import into the OCL system.

In [23]:
# Create Container Concepts and Assign Parents

def create_container_concepts():
    """
    Creates ROOT and Container concepts for organizing the LOINC hierarchy.
    Returns a list of container concept dictionaries.
    """
    container_concepts = []
    
    # 1. Create the ROOT concept
    root_concept = {
        'type': 'Concept',
        'id': 'ROOT',
        'concept_class': 'Root',
        'datatype': 'N/A',
        'source': 'LOINC',
        'owner_type': 'Organization',
        'owner': 'Regenstrief',
        'retired': False,
        'names': [
            {
                'name': 'LOINC Root Concept',
                'name_type': 'Fully-Specified',
                'locale': 'en',
                'locale_preferred': True
            },
            {
                'name': 'LOINC Root',
                'name_type': 'Short',
                'locale': 'en',
                'locale_preferred': False
            }
        ],
        'descriptions': [
            {
                'description': 'Root concept for the LOINC terminology hierarchy',
                'locale': 'en',
                'description_type': 'Full'
            }
        ],
        'extras': {
            'Code_Type': 'Root Container',
            'Code_in_Source': 'ROOT'
        }
        # Note: ROOT has no parent_concept_urls - it's the top of the hierarchy
    }
    container_concepts.append(root_concept)
    
    # 2. Define Container concepts for each concept class
    container_definitions = {
        'LOINC': {
            'id': 'LOINC_CONTAINER',
            'name': 'LOINC Terms Container',
            'short_name': 'LOINC Terms',
            'description': 'Container for all LOINC laboratory terms and observations'
        },
        'LOINC Part': {
            'id': 'LOINC_PARTS_CONTAINER', 
            'name': 'LOINC Parts Container',
            'short_name': 'LOINC Parts',
            'description': 'Container for all LOINC component parts used to build LOINC terms'
        },
        'LOINC Answer': {
            'id': 'LOINC_ANSWERS_CONTAINER',
            'name': 'LOINC Answers Container', 
            'short_name': 'LOINC Answers',
            'description': 'Container for all LOINC answer concepts used in answer lists'
        },
        'Answer List': {
            'id': 'ANSWER_LISTS_CONTAINER',
            'name': 'Answer Lists Container',
            'short_name': 'Answer Lists', 
            'description': 'Container for all LOINC answer lists and value sets'
        }
    }
    
    # 3. Create Container concepts
    for concept_class, container_info in container_definitions.items():
        container_concept = {
            'type': 'Concept',
            'id': container_info['id'],
            'concept_class': 'Container',
            'datatype': 'N/A', 
            'source': 'LOINC',
            'owner_type': 'Organization',
            'owner': 'Regenstrief',
            'retired': False,
            'names': [
                {
                    'name': container_info['name'],
                    'name_type': 'Fully-Specified',
                    'locale': 'en',
                    'locale_preferred': True
                },
                {
                    'name': container_info['short_name'],
                    'name_type': 'Short', 
                    'locale': 'en',
                    'locale_preferred': False
                }
            ],
            'descriptions': [
                {
                    'description': container_info['description'],
                    'locale': 'en',
                    'description_type': 'Full'
                }
            ],
            'parent_concept_urls': [CONCEPT_URL_PREFIX.format('ROOT')],
            'extras': {
                'Code_Type': 'Container',
                'Code_in_Source': container_info['id'],
                'Container_For': concept_class
            }
        }
        container_concepts.append(container_concept)
    
    return container_concepts

def assign_orphaned_concepts_to_containers(df):
    """
    Assigns concepts without parents to appropriate Container concepts.
    Updates the parent_concept_urls for orphaned concepts.
    """
    # Mapping of concept classes to their container IDs
    concept_class_to_container = {
        'LOINC': 'LOINC_CONTAINER',
        'LOINC Part': 'LOINC_PARTS_CONTAINER', 
        'LOINC Answer': 'LOINC_ANSWERS_CONTAINER',
        'Answer List': 'ANSWER_LISTS_CONTAINER'
    }
    
    # Create a copy to avoid modifying the original dataframe
    df_updated = df.copy()
    
    # Find concepts without parent_concept_urls
    orphaned_mask = df_updated['parent_concept_urls'].isnull()
    orphaned_concepts = df_updated[orphaned_mask]
    
    print(f"Found {len(orphaned_concepts)} orphaned concepts to assign to containers:")
    
    # Group orphaned concepts by concept_class and show counts
    orphaned_by_class = orphaned_concepts.groupby('concept_class').size()
    for concept_class, count in orphaned_by_class.items():
        container_id = concept_class_to_container.get(concept_class, 'UNKNOWN')
        print(f"  - {concept_class}: {count} concepts → {container_id}")
    
    # Assign orphaned concepts to containers
    for idx, row in orphaned_concepts.iterrows():
        concept_class = row['concept_class']
        container_id = concept_class_to_container.get(concept_class)
        
        if container_id:
            # Assign to the appropriate container
            df_updated.at[idx, 'parent_concept_urls'] = [CONCEPT_URL_PREFIX.format(container_id)]
        else:
            # If no specific container, assign to ROOT
            print(f"Warning: No container defined for concept_class '{concept_class}', assigning to ROOT")
            df_updated.at[idx, 'parent_concept_urls'] = [CONCEPT_URL_PREFIX.format('ROOT')]
    
    return df_updated

def integrate_container_concepts(concepts_df, container_concepts_list):
    """
    Integrates the container concepts into the main concepts dataframe.
    Checks for existing container concepts to prevent duplicates.
    """
    # Check if container concepts already exist
    existing_container_ids = set()
    for container_concept in container_concepts_list:
        container_id = container_concept['id']
        if container_id in concepts_df['id'].values:
            existing_container_ids.add(container_id)
    
    if existing_container_ids:
        print(f"ℹ️  Skipping {len(existing_container_ids)} container concepts that already exist: {existing_container_ids}")
        # Filter out existing containers
        new_container_concepts = [c for c in container_concepts_list if c['id'] not in existing_container_ids]
    else:
        new_container_concepts = container_concepts_list
    
    if not new_container_concepts:
        print("ℹ️  No new container concepts to add.")
        return concepts_df
    
    # Convert new container concepts to DataFrame
    container_df = pd.DataFrame(new_container_concepts)
    
    # Ensure all columns exist in both dataframes
    all_columns = set(concepts_df.columns) | set(container_df.columns)
    
    # Add missing columns to both dataframes
    for col in all_columns:
        if col not in concepts_df.columns:
            concepts_df[col] = None
        if col not in container_df.columns:
            container_df[col] = None
    
    # Reorder columns to match
    container_df = container_df[concepts_df.columns]
    
    # Concatenate the dataframes
    integrated_df = pd.concat([concepts_df, container_df], ignore_index=True)
    
    print(f"ℹ️  Added {len(new_container_concepts)} new container concepts.")
    
    return integrated_df

# Execute the container creation process
print("=== Creating Container Concepts ===")

# Check if container concepts already exist
existing_root = 'ROOT' in merged_loinc_cui_df['id'].values
existing_containers = any(cid in merged_loinc_cui_df['id'].values 
                         for cid in ['LOINC_CONTAINER', 'LOINC_PARTS_CONTAINER', 
                                   'LOINC_ANSWERS_CONTAINER', 'ANSWER_LISTS_CONTAINER'])

if existing_root and existing_containers:
    print("ℹ️  Container concepts already exist. Skipping container creation.")
    print("ℹ️  If you want to recreate containers, restart the kernel and run from the beginning.")
    # Use existing dataframe
    concepts_with_containers_df = merged_loinc_cui_df.copy()
else:
    # 1. Create container concepts
    container_concepts_list = create_container_concepts()
    print(f"Created {len(container_concepts_list)} container concepts (including ROOT)")

    # 2. Assign orphaned concepts to containers (work with a copy)
    concepts_with_parents_df = assign_orphaned_concepts_to_containers(merged_loinc_cui_df)

    # 3. Integrate container concepts into the main dataframe
    concepts_with_containers_df = integrate_container_concepts(concepts_with_parents_df, container_concepts_list)

    print(f"\nFinal concept count: {len(concepts_with_containers_df)}")
    print(f"Container concepts: {len([c for c in container_concepts_list if c['id'] not in merged_loinc_cui_df['id'].values])}")
    print(f"Original concepts: {len(merged_loinc_cui_df)}")

# 4. Final validation - check for remaining orphaned concepts (excluding ROOT)
remaining_orphaned = concepts_with_containers_df[
    (concepts_with_containers_df['parent_concept_urls'].isnull()) & 
    (concepts_with_containers_df['id'] != 'ROOT')
]
if len(remaining_orphaned) == 0:
    print("✅ SUCCESS: All concepts now have parent assignments!")
else:
    print(f"⚠️  WARNING: {len(remaining_orphaned)} concepts still without parents:")
    print(remaining_orphaned[['id', 'concept_class']].head())

# 5. Sort concepts hierarchically so parents come before children
def sort_concepts_hierarchically(df):
    """
    Sorts concepts so that parent concepts appear before their children.
    Uses topological sorting based on parent_concept_urls.
    """
    print("Sorting concepts hierarchically...")
    
    # Create a copy to work with
    df_sorted = df.copy().reset_index(drop=True)
    
    # Extract parent IDs from parent_concept_urls
    def extract_parent_ids(parent_urls):
        # Handle None or NaN
        if parent_urls is None:
            return []
        
        # For scalar values, check if NaN
        if not isinstance(parent_urls, (list, tuple, np.ndarray)):
            try:
                if pd.isna(parent_urls):
                    return []
            except (TypeError, ValueError):
                pass
            # If it's a string, return empty (shouldn't happen in normal flow)
            if isinstance(parent_urls, str):
                return []
            # If it's some other scalar, return empty
            return []
        
        # For array-like objects, check if empty
        if len(parent_urls) == 0:
            return []
        
        # Continue with extraction logic
        try:
            # Extract IDs from URLs like "/orgs/Regenstrief/sources/LOINC/concepts/ROOT/"
            parent_ids = []
            for url in parent_urls:
                if isinstance(url, str):
                    # Extract the concept ID from the URL
                    parts = url.strip('/').split('/')
                    if len(parts) >= 5:  # Expected format: orgs/Regenstrief/sources/LOINC/concepts/ID
                        parent_ids.append(parts[-1])
            return parent_ids
        except:
            return []
    
    df_sorted['parent_ids'] = df_sorted['parent_concept_urls'].apply(extract_parent_ids)
    
    # Topological sort
    sorted_concepts = []
    processed_ids = set()
    remaining_df = df_sorted.copy()
    
    iteration = 0
    max_iterations = len(df_sorted) + 10  # Safety limit
    
    while len(remaining_df) > 0 and iteration < max_iterations:
        iteration += 1
        
        # Find concepts whose parents are already processed (or have no parents)
        ready_concepts = []
        
        for idx, row in remaining_df.iterrows():
            parent_ids = row['parent_ids']
            
            # Concept is ready if it has no parents or all parents are already processed
            if not parent_ids or all(pid in processed_ids for pid in parent_ids):
                ready_concepts.append(idx)
        
        if not ready_concepts:
            # If no concepts are ready, we might have circular dependencies
            # Add remaining concepts in order of their dependency count
            print(f"Warning: Possible circular dependencies detected. Adding remaining {len(remaining_df)} concepts by dependency count.")
            dependency_counts = remaining_df['parent_ids'].apply(lambda x: len([p for p in x if p not in processed_ids]))
            ready_concepts = dependency_counts.sort_values().index.tolist()
        
        # Add ready concepts to sorted list
        for idx in ready_concepts:
            row = remaining_df.loc[idx]
            sorted_concepts.append(row)
            processed_ids.add(row['id'])
        
        # Remove processed concepts from remaining
        remaining_df = remaining_df.drop(ready_concepts)
        
        if iteration % 50 == 0:
            print(f"  Processed {len(sorted_concepts)}/{len(df_sorted)} concepts...")
    
    # Create the final sorted dataframe
    final_sorted_df = pd.DataFrame(sorted_concepts).reset_index(drop=True)
    
    # Drop the temporary parent_ids column
    final_sorted_df = final_sorted_df.drop('parent_ids', axis=1)
    
    print(f"Hierarchical sorting complete. Processed {len(final_sorted_df)} concepts in {iteration} iterations.")
    
    return final_sorted_df

# Debug code to identify circular dependencies - FIXED VERSION
def debug_circular_dependencies(df):
    """
    Analyzes the concepts dataframe to identify potential circular dependencies.
    """
    print("\n=== DEBUG: Analyzing Circular Dependencies ===")
    
    # Extract parent IDs from parent_concept_urls (FIXED VERSION)
    def extract_parent_ids(parent_urls):
        # Handle None or NaN
        if parent_urls is None:
            return []
        
        # For scalar values, check if NaN
        if not isinstance(parent_urls, (list, tuple, np.ndarray)):
            try:
                if pd.isna(parent_urls):
                    return []
            except (TypeError, ValueError):
                pass
            # If it's a string, return empty (shouldn't happen in normal flow)
            if isinstance(parent_urls, str):
                return []
            # If it's some other scalar, return empty
            return []
        
        # For array-like objects, check if empty
        if len(parent_urls) == 0:
            return []
        
        # Continue with extraction logic
        try:
            parent_ids = []
            for url in parent_urls:
                if isinstance(url, str):
                    parts = url.strip('/').split('/')
                    if len(parts) >= 5:
                        parent_ids.append(parts[-1])
            return parent_ids
        except:
            return []
    
    df_debug = df.copy()
    df_debug['parent_ids'] = df_debug['parent_concept_urls'].apply(extract_parent_ids)
    
    # Create a dictionary of concept -> parents mapping
    concept_parents = {}
    for _, row in df_debug.iterrows():
        concept_id = row['id']
        parent_ids = row['parent_ids']
        concept_parents[concept_id] = parent_ids
    
    # Find concepts that are involved in cycles
    def find_path_to_concept(start_concept, target_concept, visited=None):
        """Returns the path if target_concept is reachable from start_concept"""
        if visited is None:
            visited = set()
        
        if start_concept in visited:
            return None  # Already visited, potential cycle
        
        if start_concept == target_concept:
            return [start_concept]
        
        visited.add(start_concept)
        
        # Check all parents of the current concept
        parents = concept_parents.get(start_concept, [])
        for parent in parents:
            if parent in concept_parents:  # Only follow parents that exist as concepts
                path = find_path_to_concept(parent, target_concept, visited.copy())
                if path:
                    return [start_concept] + path
        
        return None
    
    # Check for cycles
    cycles_found = []
    concepts_in_cycles = set()
    
    for concept_id in concept_parents.keys():
        # Check if this concept can reach itself through its parents
        parents = concept_parents.get(concept_id, [])
        for parent in parents:
            if parent in concept_parents:
                cycle_path = find_path_to_concept(parent, concept_id)
                if cycle_path:
                    full_cycle = [concept_id] + cycle_path
                    cycles_found.append(full_cycle)
                    concepts_in_cycles.update(full_cycle)
    
    if cycles_found:
        print(f"🚨 Found {len(cycles_found)} circular dependency cycles:")
        for i, cycle in enumerate(cycles_found, 1):
            print(f"  Cycle {i}: {' → '.join(cycle)} → {cycle[0]}")
    
    # Identify concepts that might be causing the sorting issues
    problem_concepts = []
    all_concept_ids = set(concept_parents.keys())
    
    # Find concepts whose parents don't exist as concepts
    for concept_id, parent_ids in concept_parents.items():
        missing_parents = [p for p in parent_ids if p not in all_concept_ids]
        if missing_parents:
            problem_concepts.append({
                'concept_id': concept_id,
                'issue': 'missing_parents',
                'details': missing_parents
            })
    
    # Find concepts with complex parent relationships
    multi_parent_concepts = []
    for concept_id, parent_ids in concept_parents.items():
        if len(parent_ids) > 1:
            multi_parent_concepts.append({
                'concept_id': concept_id,
                'parent_count': len(parent_ids),
                'parents': parent_ids
            })
    
    print(f"\n📊 Dependency Analysis Summary:")
    print(f"   Total concepts: {len(concept_parents)}")
    print(f"   Concepts in cycles: {len(concepts_in_cycles)}")
    print(f"   Concepts with missing parents: {len(problem_concepts)}")
    print(f"   Concepts with multiple parents: {len(multi_parent_concepts)}")
    
    if problem_concepts:
        print(f"\n⚠️  Concepts with missing parents:")
        for problem in problem_concepts[:5]:  # Show first 5
            print(f"   {problem['concept_id']} → missing: {problem['details']}")
        if len(problem_concepts) > 5:
            print(f"   ... and {len(problem_concepts) - 5} more")
    
    if multi_parent_concepts:
        print(f"\n📋 Concepts with multiple parents:")
        for concept in multi_parent_concepts[:5]:  # Show first 5
            print(f"   {concept['concept_id']} → parents: {concept['parents']}")
        if len(multi_parent_concepts) > 5:
            print(f"   ... and {len(multi_parent_concepts) - 5} more")
    
    # Show the specific concepts that are likely causing the sorting warning
    if concepts_in_cycles:
        print(f"\n🔍 Concepts likely causing the sorting warning:")
        cycle_concept_details = df_debug[df_debug['id'].isin(concepts_in_cycles)][['id', 'concept_class', 'parent_ids']]
        print(cycle_concept_details.to_string(index=False))
    
    print("=== End Debug Analysis ===\n")
    
    return concepts_in_cycles, problem_concepts, multi_parent_concepts

# Run the debug analysis
debug_results = debug_circular_dependencies(concepts_with_containers_df)

# Fix missing parent references by reassigning to containers
def fix_missing_parent_references(df):
    """
    Identifies concepts with missing parent references and reassigns them to appropriate containers.
    """
    print("\n=== FIXING: Missing Parent References ===")
    
    # Get all concept IDs that exist in our dataset
    existing_concept_ids = set(df['id'].values)
    
    # Container mapping
    concept_class_to_container = {
        'LOINC': 'LOINC_CONTAINER',
        'LOINC Part': 'LOINC_PARTS_CONTAINER', 
        'LOINC Answer': 'LOINC_ANSWERS_CONTAINER',
        'Answer List': 'ANSWER_LISTS_CONTAINER'
    }
    
    # Extract parent IDs function (same logic as above)
    def extract_parent_ids(parent_urls):
        # Handle None or NaN
        if parent_urls is None:
            return []
        
        # For scalar values, check if NaN
        if not isinstance(parent_urls, (list, tuple, np.ndarray)):
            try:
                if pd.isna(parent_urls):
                    return []
            except (TypeError, ValueError):
                pass
            # If it's a string, return empty (shouldn't happen in normal flow)
            if isinstance(parent_urls, str):
                return []
            # If it's some other scalar, return empty
            return []
        
        # For array-like objects, check if empty
        if len(parent_urls) == 0:
            return []
        
        # Continue with extraction logic
        try:
            parent_ids = []
            for url in parent_urls:
                if isinstance(url, str):
                    parts = url.strip('/').split('/')
                    if len(parts) >= 5:
                        parent_ids.append(parts[-1])
            return parent_ids
        except:
            return []
    
    df_fixed = df.copy()
    concepts_fixed = []
    
    for idx, row in df_fixed.iterrows():
            concept_id = row['id']
            parent_urls = row['parent_concept_urls']
            concept_class = row['concept_class']
            
            # Use extract_parent_ids to safely handle all parent_urls formats
            parent_ids = extract_parent_ids(parent_urls)
            
            if parent_ids:  # Only proceed if we have parent IDs to check
                missing_parents = [pid for pid in parent_ids if pid not in existing_concept_ids]
                
                if missing_parents:
                    # This concept has missing parent references
                    # Reassign to appropriate container
                    container_id = concept_class_to_container.get(concept_class, 'ROOT')
                    new_parent_url = CONCEPT_URL_PREFIX.format(container_id)
                    
                    # Update the parent_concept_urls
                    df_fixed.at[idx, 'parent_concept_urls'] = [new_parent_url]
                    
                    concepts_fixed.append({
                        'concept_id': concept_id,
                        'concept_class': concept_class,
                        'missing_parents': missing_parents,
                        'new_parent': container_id
                    })
    
    print(f"✅ Fixed {len(concepts_fixed)} concepts with missing parent references")
    print("=== End Fix ===\n")
    
    return df_fixed, concepts_fixed

# Apply the fix
concepts_with_containers_df_fixed, fixed_concepts = fix_missing_parent_references(concepts_with_containers_df)

# Update the variable name for clarity
concepts_with_containers_df = concepts_with_containers_df_fixed

# Apply hierarchical sorting
final_concepts_with_hierarchy_df = sort_concepts_hierarchically(concepts_with_containers_df)

# 6. Update the global dataframe with the final result
merged_loinc_cui_df = final_concepts_with_hierarchy_df.copy()

print("=== Container Concepts Creation Complete ===")

=== Creating Container Concepts ===
Created 5 container concepts (including ROOT)


Found 46849 orphaned concepts to assign to containers:
  - Answer List: 4621 concepts → ANSWER_LISTS_CONTAINER
  - LOINC Part: 42228 concepts → LOINC_PARTS_CONTAINER


C:\Users\jamlung\AppData\Local\Temp\1\ipykernel_32676\934588327.py:203: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  integrated_df = pd.concat([concepts_df, container_df], ignore_index=True)


ℹ️  Added 5 new container concepts.

Final concept count: 202071
Container concepts: 5
Original concepts: 202066
✅ SUCCESS: All concepts now have parent assignments!

=== DEBUG: Analyzing Circular Dependencies ===

📊 Dependency Analysis Summary:
   Total concepts: 202071
   Concepts in cycles: 0
   Concepts with missing parents: 107347
   Concepts with multiple parents: 621

⚠️  Concepts with missing parents:
   1-8 → missing: ['LP380309-7']
   10-9 → missing: ['LP380471-5']
   100-8 → missing: ['LP380552-2']
   1000-9 → missing: ['LP248770-2']
   10000-8 → missing: ['LP407088-6']
   ... and 107342 more

📋 Concepts with multiple parents:
   LP100617-2 → parents: ['LP31465-5', 'LP7840-4']
   LP101975-3 → parents: ['LP7779-4', 'LP7782-8', 'LP7778-6']
   LP101977-9 → parents: ['LP29672-0', 'LP7778-6']
   LP101982-9 → parents: ['LP172861-9', 'LP7840-4']
   LP113911-4 → parents: ['LP31406-9', 'LP111174-1']
   ... and 616 more
=== End Debug Analysis ===


=== FIXING: Missing Parent Reference

In [24]:
# Quick cleanup of dfs before output
# Concepts - change type of 'extras.VersionFirstReleased' to string
merged_loinc_cui_df['extras.VersionLastChanged'] = merged_loinc_cui_df['extras.VersionLastChanged'].astype(str).replace('nan', '')
merged_loinc_cui_df.drop(columns=["parent_concept","match-id"], inplace=True)


# Mappings - remove bad values:
for mapping in all_mappings:
    if "extras" in mapping and mapping["extras"]:
        # Create a list of keys to remove to avoid modifying the dictionary while iterating
        keys_to_remove = [
            key for key, value in mapping["extras"].items()
            if value is None or (isinstance(value, float) and pd.isna(value))
        ]
        for key in keys_to_remove:
            del mapping["extras"][key]

# FINAL VALIDATION - all concepts have a parent_concept_urls (except Root)
print("--- Starting Final Validation ---")
# Check if any concept is missing parent_concept_urls (excluding Root concept)
missing_parent_concepts = merged_loinc_cui_df[
    (merged_loinc_cui_df['parent_concept_urls'].isnull()) & 
    (merged_loinc_cui_df['id'] != 'ROOT')
]
if not missing_parent_concepts.empty:
    print("ERROR: The following concepts are missing a parent_concept_url:")
    print(missing_parent_concepts[['id']]) # Adjust column names as needed
else:
    print("SUCCESS: All concepts have a parent_concept_url (Root concept excluded from check).")
print("--- Final Validation Complete ---")


--- Starting Final Validation ---
SUCCESS: All concepts have a parent_concept_url (Root concept excluded from check).
--- Final Validation Complete ---


In [ ]:
# Output - Updated with True Hierarchical Chunking

output_folder_path = 'output'
os.makedirs(output_folder_path, exist_ok=True)

CHUNK_SIZE = 50000

def is_valid_value(value):
    """
    Checks if a value is not NaN, None, an empty list/dict, or an empty string.
    Handles non-scalar values (like lists and pandas Series) safely.
    """
    if value is None:
        return False
    
    # Handle empty string specifically
    if isinstance(value, str) and value.strip() == "":
        return False

    # Handle scalar values first
    if not isinstance(value, (list, tuple, pd.Series, np.ndarray)):
        return not pd.isna(value)
    
    # Handle non-scalar values (lists, Series, etc.)
    # Consider them valid only if they contain at least one non-NaN value
    if isinstance(value, pd.Series):
        return not value.dropna().empty
    elif isinstance(value, (list, np.ndarray)):
        # Check for empty list/array or list/array of NaNs
        return any(pd.notna(v) for v in value)
    
    # For other types, check if they are "empty"
    if hasattr(value, '__len__') and len(value) == 0:
        return False

    return True

def safe_na_to_none(value):
    """
    Safely converts NaN values to None, handling both scalar and array-like values.
    This prevents the ValueError: The truth value of an array with more than one element is ambiguous.
    """
    # Handle None explicitly
    if value is None:
        return None
    
    # Handle scalar values (numbers, strings, single values)
    if not isinstance(value, (list, tuple, pd.Series, np.ndarray)):
        try:
            return None if pd.isna(value) else value
        except (TypeError, ValueError):
            # If pd.isna() fails for any reason, return the original value
            return value
    
    # Handle array-like values (lists, arrays, Series)
    # For array-like values, we don't want to convert them to None
    # unless they are completely empty or all NaN
    if isinstance(value, (list, tuple)):
        if len(value) == 0:
            return None
        # Check if all elements are NaN
        try:
            if all(pd.isna(v) for v in value):
                return None
            else:
                return value
        except (TypeError, ValueError):
            # If checking for NaN fails, keep the original value
            return value
    
    elif isinstance(value, pd.Series):
        if value.empty or value.dropna().empty:
            return None
        else:
            return value
    
    elif isinstance(value, np.ndarray):
        if value.size == 0:
            return None
        try:
            if np.all(pd.isna(value)):
                return None
            else:
                return value
        except (TypeError, ValueError):
            return value
    
    # For any other type, return as-is
    return value

def extract_parent_ids_from_urls(parent_urls):
    """
    Extracts concept IDs from parent_concept_urls.
    
    Args:
        parent_urls: Can be None, NaN, a string, or a list/array of URL strings
        
    Returns:
        list: List of concept IDs extracted from the URLs, empty list if no valid URLs found
    """
    # Handle None or NaN values
    if parent_urls is None:
        return []
    
    # Handle scalar values (single values, not lists/arrays)
    if not isinstance(parent_urls, (list, tuple, np.ndarray, pd.Series)):
        try:
            # Check if it's a pandas NaN value
            if pd.isna(parent_urls):
                return []
        except (TypeError, ValueError):
            # If pd.isna() fails, it's likely not a pandas-compatible type
            pass
        
        # If it's a string (shouldn't happen in normal flow), return empty
        if isinstance(parent_urls, str):
            return []
        
        # For any other scalar type, return empty
        return []
    
    # Handle empty collections
    if len(parent_urls) == 0:
        return []
    
    # Extract concept IDs from the URLs
    parent_ids = []
    try:
        for url in parent_urls:
            # Skip None, NaN, or non-string values
            if url is None or not isinstance(url, str):
                continue
                
            # Skip empty strings
            if not url.strip():
                continue
            
            try:
                # Extract the concept ID from URL format: "/orgs/Regenstrief/sources/LOINC/concepts/ID/"
                parts = url.strip('/').split('/')
                if len(parts) >= 5 and parts[-2] == 'concepts':
                    concept_id = parts[-1]
                    if concept_id:  # Make sure we don't add empty strings
                        parent_ids.append(concept_id)
            except (IndexError, AttributeError):
                # Skip malformed URLs
                continue
                
    except Exception:
        # If any unexpected error occurs during iteration, return what we have so far
        pass
    
    return parent_ids

def organize_concepts_by_dependency_levels(data_list):
    """
    Organizes concepts into dependency levels where each level contains concepts
    whose parents are all in previous levels.
    
    Returns a list of lists, where each inner list represents a dependency level.
    """
    print("Organizing concepts by dependency levels...")
    
    # Convert to DataFrame if it's a list of dictionaries
    if isinstance(data_list, list):
        df = pd.DataFrame(data_list)
    else:
        df = data_list.copy()
    
    # Create a mapping of concept ID to concept data (convert Series to dict)
    concept_map = {row['id']: row.to_dict() for _, row in df.iterrows()}
    all_concept_ids = set(concept_map.keys())
    
    # Extract parent relationships
    concept_parents = {}
    for concept_id, concept_data in concept_map.items():
        parent_urls = concept_data.get('parent_concept_urls', [])
        parent_ids = extract_parent_ids_from_urls(parent_urls)
        # Only include parents that exist in our dataset
        valid_parent_ids = [pid for pid in parent_ids if pid in all_concept_ids]
        concept_parents[concept_id] = valid_parent_ids
    
    # Organize into levels
    levels = []
    processed_concepts = set()
    remaining_concepts = set(all_concept_ids)
    
    level_num = 1
    while remaining_concepts:
        print(f"  Processing level {level_num}...")
        
        # Find concepts that can be processed at this level
        # (concepts whose parents are all already processed or have no parents)
        current_level_concepts = []
        
        for concept_id in list(remaining_concepts):
            parents = concept_parents.get(concept_id, [])
            
            # Concept can be processed if:
            # 1. It has no parents, OR
            # 2. All its parents have already been processed
            if not parents or all(parent_id in processed_concepts for parent_id in parents):
                current_level_concepts.append(concept_map[concept_id])
                remaining_concepts.remove(concept_id)
                processed_concepts.add(concept_id)
        
        if not current_level_concepts:
            # No progress made - might have circular dependencies
            print(f"    Warning: No progress at level {level_num}. Remaining concepts: {len(remaining_concepts)}")
            print(f"    Adding remaining concepts to current level to break potential cycles...")
            
            # Add remaining concepts to break the deadlock
            for concept_id in list(remaining_concepts):
                current_level_concepts.append(concept_map[concept_id])
                processed_concepts.add(concept_id)
            remaining_concepts.clear()
        
        if current_level_concepts:
            levels.append(current_level_concepts)
            print(f"    Level {level_num}: {len(current_level_concepts)} concepts")
            
            # Show some examples of what's in this level
            if level_num <= 5:  # Only show details for first few levels
                concept_classes = {}
                for concept in current_level_concepts:
                    concept_class = concept.get('concept_class', 'Unknown')
                    concept_classes[concept_class] = concept_classes.get(concept_class, 0) + 1
                class_summary = ', '.join([f"{cls}: {count}" for cls, count in concept_classes.items()])
                print(f"      Content: {class_summary}")
        
        level_num += 1
        
        # Safety break to prevent infinite loops
        if level_num > 50:
            print(f"    Warning: Stopped at level {level_num} to prevent infinite loop")
            break
    
    print(f"  Total dependency levels: {len(levels)}")
    return levels

def clean_concept_for_json(concept_dict):
    """
    Cleans a concept dictionary by removing attributes with invalid values (NaN, None, empty, etc.)
    before writing to JSON. This ensures clean JSON output without empty/null attributes.
    """
    if not isinstance(concept_dict, dict):
        return concept_dict
    
    cleaned_dict = {}
    
    for key, value in concept_dict.items():
        if is_valid_value(value):
            # Special handling for nested structures
            if key == 'extras' and isinstance(value, dict):
                # Clean extras dict and only include if it has valid content
                cleaned_extras = {k: v for k, v in value.items() if is_valid_value(v)}
                if cleaned_extras:  # Only add extras if it contains valid data
                    cleaned_dict[key] = cleaned_extras
            elif key in ['names', 'descriptions'] and isinstance(value, list):
                # Clean lists of names/descriptions
                cleaned_list = []
                for item in value:
                    if isinstance(item, dict):
                        cleaned_item = {k: v for k, v in item.items() if is_valid_value(v)}
                        if cleaned_item:  # Only add if the cleaned item has content
                            cleaned_list.append(cleaned_item)
                    elif is_valid_value(item):
                        cleaned_list.append(item)
                if cleaned_list:  # Only add the list if it has valid content
                    cleaned_dict[key] = cleaned_list
            elif key == 'parent_concept_urls' and isinstance(value, list):
                # Clean parent_concept_urls list
                cleaned_urls = [url for url in value if is_valid_value(url)]
                if cleaned_urls:  # Only add if there are valid URLs
                    cleaned_dict[key] = cleaned_urls
            else:
                # For all other attributes, just include if valid
                cleaned_dict[key] = value
    
    return cleaned_dict

def write_hierarchical_chunks(data_list, file_prefix):
    """
    Chunks a list of dictionaries by true dependency levels and writes each chunk to separate JSON Lines files.
    Each level contains concepts whose parents are all in previous levels.
    
    FIXED: Added UTF-8 encoding and ensure_ascii=False for proper Unicode handling.
    """
    print(f"Writing {len(data_list)} records for '{file_prefix}' to hierarchical dependency-based chunked files...")
    
    # Organize concepts by dependency levels
    dependency_levels = organize_concepts_by_dependency_levels(data_list)
    
    chunk_number = 1
    
    # PHASE 1: Handle ROOT concepts first (must be in first chunk(s))
    root_concepts = []
    container_concepts = []
    remaining_levels = []
    
    for level_num, level_concepts in enumerate(dependency_levels, 1):
        # Check if this level contains ROOT concepts
        if any(c.get('concept_class') == 'Root' for c in level_concepts):
            root_concepts = level_concepts
            print(f"Found ROOT level: Level {level_num} with {len(root_concepts)} concepts")
        # Check if this level contains Container concepts
        elif any(c.get('concept_class') == 'Container' for c in level_concepts):
            container_concepts = level_concepts
            print(f"Found Container level: Level {level_num} with {len(container_concepts)} concepts")
        else:
            # All other levels
            remaining_levels.append((level_num, level_concepts))
    
    # Write ROOT concepts first
    if root_concepts:
        print(f"Writing ROOT concepts ({len(root_concepts)} concepts)...")
        
        # ROOT concepts should fit in one chunk, but handle if they don't
        if len(root_concepts) <= CHUNK_SIZE:
            output_file_name = f"{file_prefix}_{chunk_number:03d}_root.json"
            output_file_path = os.path.join(output_folder_path, output_file_name)
            
            print(f"  Writing ROOT - Chunk {chunk_number} to {output_file_path}...")
            # FIXED: Added encoding='utf-8' and ensure_ascii=False
            with open(output_file_path, 'w', encoding='utf-8') as f:
                for record in root_concepts:
                    cleaned_record = clean_concept_for_json(record)
                    json.dump(cleaned_record, f, ensure_ascii=False)
                    f.write('\n')
            chunk_number += 1
        else:
            # Handle case where ROOT concepts exceed chunk size (unlikely but defensive)
            num_chunks_needed = (len(root_concepts) - 1) // CHUNK_SIZE + 1
            print(f"  ROOT concepts require {num_chunks_needed} chunks...")
            
            for i in range(0, len(root_concepts), CHUNK_SIZE):
                chunk = root_concepts[i:i + CHUNK_SIZE]
                chunk_in_group = i // CHUNK_SIZE + 1
                
                output_file_name = f"{file_prefix}_{chunk_number:03d}_root_part{chunk_in_group}.json"
                output_file_path = os.path.join(output_folder_path, output_file_name)
                
                print(f"    Writing ROOT Part {chunk_in_group}/{num_chunks_needed} - Chunk {chunk_number} to {output_file_path}...")
                # FIXED: Added encoding='utf-8' and ensure_ascii=False
                with open(output_file_path, 'w', encoding='utf-8') as f:
                    for record in chunk:
                        cleaned_record = clean_concept_for_json(record)
                        json.dump(cleaned_record, f, ensure_ascii=False)
                        f.write('\n')
                chunk_number += 1
    
    # PHASE 2: Write Container concepts second
    if container_concepts:
        print(f"Writing Container concepts ({len(container_concepts)} concepts)...")
        
        # Container concepts should typically fit in one chunk, but handle if they don't
        if len(container_concepts) <= CHUNK_SIZE:
            output_file_name = f"{file_prefix}_{chunk_number:03d}_containers.json"
            output_file_path = os.path.join(output_folder_path, output_file_name)
            
            print(f"  Writing Containers - Chunk {chunk_number} to {output_file_path}...")
            # FIXED: Added encoding='utf-8' and ensure_ascii=False
            with open(output_file_path, 'w', encoding='utf-8') as f:
                for record in container_concepts:
                    cleaned_record = clean_concept_for_json(record)
                    json.dump(cleaned_record, f, ensure_ascii=False)
                    f.write('\n')
            chunk_number += 1
        else:
            # Handle case where Container concepts exceed chunk size
            num_chunks_needed = (len(container_concepts) - 1) // CHUNK_SIZE + 1
            print(f"  Container concepts require {num_chunks_needed} chunks...")
            
            for i in range(0, len(container_concepts), CHUNK_SIZE):
                chunk = container_concepts[i:i + CHUNK_SIZE]
                chunk_in_group = i // CHUNK_SIZE + 1
                
                output_file_name = f"{file_prefix}_{chunk_number:03d}_containers_part{chunk_in_group}.json"
                output_file_path = os.path.join(output_folder_path, output_file_name)
                
                print(f"    Writing Containers Part {chunk_in_group}/{num_chunks_needed} - Chunk {chunk_number} to {output_file_path}...")
                # FIXED: Added encoding='utf-8' and ensure_ascii=False
                with open(output_file_path, 'w', encoding='utf-8') as f:
                    for record in chunk:
                        cleaned_record = clean_concept_for_json(record)
                        json.dump(cleaned_record, f, ensure_ascii=False)
                        f.write('\n')
                chunk_number += 1
    
    # PHASE 3: Write remaining levels one by one
    for level_num, level_concepts in remaining_levels:
        level_name = f"level_{level_num:02d}"
        print(f"Writing Level {level_num} ({len(level_concepts)} concepts)...")
        
        # If level fits in one chunk, write it directly
        if len(level_concepts) <= CHUNK_SIZE:
            output_file_name = f"{file_prefix}_{chunk_number:03d}_{level_name}.json"
            output_file_path = os.path.join(output_folder_path, output_file_name)
            
            print(f"  Writing Level {level_num} - Chunk {chunk_number} to {output_file_path}...")
            # FIXED: Added encoding='utf-8' and ensure_ascii=False
            with open(output_file_path, 'w', encoding='utf-8') as f:
                for record in level_concepts:
                    cleaned_record = clean_concept_for_json(record)
                    json.dump(cleaned_record, f, ensure_ascii=False)
                    f.write('\n')
            chunk_number += 1
        
        # If level is too large, split into multiple chunks
        else:
            num_chunks_in_level = (len(level_concepts) - 1) // CHUNK_SIZE + 1
            print(f"  Level {level_num} requires {num_chunks_in_level} chunks...")
            
            for i in range(0, len(level_concepts), CHUNK_SIZE):
                chunk = level_concepts[i:i + CHUNK_SIZE]
                chunk_in_level = i // CHUNK_SIZE + 1
                
                output_file_name = f"{file_prefix}_{chunk_number:03d}_{level_name}_part{chunk_in_level}.json"
                output_file_path = os.path.join(output_folder_path, output_file_name)
                
                print(f"    Writing Level {level_num} Part {chunk_in_level}/{num_chunks_in_level} - Chunk {chunk_number} to {output_file_path}...")
                # FIXED: Added encoding='utf-8' and ensure_ascii=False
                with open(output_file_path, 'w', encoding='utf-8') as f:
                    for record in chunk:
                        cleaned_record = clean_concept_for_json(record)
                        json.dump(cleaned_record, f, ensure_ascii=False)
                        f.write('\n')
                chunk_number += 1
    
    print(f"Hierarchical chunking complete. Generated {chunk_number - 1} total chunks.")
    print("\nChunking strategy summary:")
    print("  - ROOT concepts: Always in first chunk(s)")
    print("  - Container concepts: Always in second chunk(s)")  
    print("  - Remaining levels: Processed sequentially, one level at a time")
    print(f"  - All chunks respect {CHUNK_SIZE:,} record limit")

def write_chunks(data_list, file_prefix):
    """
    Original chunking function - chunks a list of dictionaries and writes each chunk to a separate JSON Lines file.
    Used for mappings which don't need hierarchical organization.
    
    FIXED: Added UTF-8 encoding and ensure_ascii=False for proper Unicode handling.
    """
    print(f"Writing {len(data_list)} records for '{file_prefix}' to chunked files...")
    for i in range(0, len(data_list), CHUNK_SIZE):
        chunk = data_list[i:i + CHUNK_SIZE]
        chunk_number = i // CHUNK_SIZE + 1
        output_file_name = f"{file_prefix}_{chunk_number:03d}.json"
        output_file_path = os.path.join(output_folder_path, output_file_name)
        
        print(f"Writing chunk {chunk_number} of {len(data_list)//CHUNK_SIZE + 1} to {output_file_path}...")
        # FIXED: Added encoding='utf-8' and ensure_ascii=False
        with open(output_file_path, 'w', encoding='utf-8') as f:
            for record in chunk:
                cleaned_record = clean_concept_for_json(record)
                json.dump(cleaned_record, f, ensure_ascii=False)
                f.write('\n')

# === New cleanup step for merged_loinc_cui_df ===
# Iterate over columns starting with 'extras.' and replace bad values with None
for col in merged_loinc_cui_df.columns:
    if col.startswith('extras.'):
        merged_loinc_cui_df[col] = merged_loinc_cui_df[col].apply(safe_na_to_none)


# Process the merged_loinc_cui_df for Concepts
concepts_data = []
print("Generating OCL Concepts from merged_loinc_cui_df...")
total_concepts = len(merged_loinc_cui_df)
for idx, row in merged_loinc_cui_df.iterrows():
    if (idx + 1) % 1000 == 0 or (idx + 1) == total_concepts:
        print(f"   Processed {idx + 1}/{total_concepts} concepts.")
    
    row_dict = row.to_dict()
    
    # Logic to dynamically process all 'names' attributes with locale_preferred flag
    names_list = []
    keys_to_remove = []
    for key, value in list(row_dict.items()):
        if key.startswith('names.'):
            if is_valid_value(value):
                # Use regex to extract name_type and locale
                match = re.match(r'names\.([^.]+)\.([^\[]+)\[\d+\]', key)
                if match:
                    name_type, locale = match.groups()
                    locale_preferred = (name_type == 'Long Common')
                    names_list.append({'name': value, 'name_type': name_type, 'locale': locale, 'locale_preferred': locale_preferred})
            del row_dict[key]
    
    if names_list:
        row_dict['names'] = names_list
    
    # The 'description' is also in a specific format in the OCL
    description = row_dict.pop('description', None)
    if is_valid_value(description):
        row_dict['descriptions'] = [{'description': description, 'locale': 'en', 'description_type': 'Full'}]
    else:
        row_dict['descriptions'] = []

    # Logic to handle the 'extras' attribute as a single dictionary at the end
    extras_dict = {}
    keys_to_remove = []
    for key, value in list(row_dict.items()):
        if key.startswith('extras.'):
            if is_valid_value(value):
                # Clean the key name by removing 'extras.' prefix
                new_key = key[len('extras.'):]
                extras_dict[new_key] = value
            del row_dict[key]
    
    if extras_dict:
        row_dict['extras'] = extras_dict

    # Final cleanup of any remaining invalid values
    cleaned_row_dict = {k: v for k, v in row_dict.items() if is_valid_value(v)}
    if 'extras' in cleaned_row_dict and not cleaned_row_dict['extras']:
        del cleaned_row_dict['extras']

    concepts_data.append(cleaned_row_dict)

# Use hierarchical chunking for concepts
write_hierarchical_chunks(concepts_data, 'concepts')

# Process the all_mappings for Mappings
mappings_data = []
print("Generating OCL Mappings from all_mappings...")
total_mappings = len(all_mappings)

all_mappings_df = pd.DataFrame(all_mappings)

# === New cleanup step for all_mappings_df ===
for col in all_mappings_df.columns:
    if col.startswith('extras.'):
        all_mappings_df[col] = all_mappings_df[col].apply(safe_na_to_none)

for idx, row in all_mappings_df.iterrows():
    if (idx + 1) % 1000 == 0 or (idx + 1) == total_mappings:
        print(f"   Processed {idx + 1}/{total_mappings} mappings.")
    
    row_dict = row.to_dict()

    # Logic to handle the 'extras' attribute as a single dictionary at the end
    extras_dict = {}
    keys_to_remove = []
    for key, value in list(row_dict.items()):
        if key.startswith('extras.'):
            if is_valid_value(value):
                # Clean the key name by removing 'extras.' prefix
                new_key = key[len('extras.'):]
                extras_dict[new_key] = value
            del row_dict[key]
    
    if extras_dict:
        row_dict['extras'] = extras_dict

    # Final cleanup of any remaining invalid values
    cleaned_row_dict = {k: v for k, v in row_dict.items() if is_valid_value(v)}
    if 'extras' in cleaned_row_dict and not cleaned_row_dict['extras']:
        del cleaned_row_dict['extras']
        
    mappings_data.append(cleaned_row_dict)

# Use regular chunking for mappings (no hierarchy needed)
write_chunks(mappings_data, 'mappings')

print("OCL Bulk Import files generated successfully with true hierarchical dependency-based chunking for concepts.")
print("\nFile organization example:")
print("  concepts_001_level_01_root.json        - Level 1: ROOT concept(s)")
print("  concepts_002_level_02_containers.json  - Level 2: Container concepts")
print("  concepts_003_level_03.json             - Level 3: Concepts with ROOT/Container parents")
print("  concepts_004_level_04_part1.json       - Level 4: Concepts with Level 3 parents (part 1)")
print("  concepts_005_level_04_part2.json       - Level 4: Concepts with Level 3 parents (part 2)")
print("  concepts_006_level_05.json             - Level 5: Concepts with Level 4 parents")
print("  ...                                    - Additional dependency levels as needed")
print("  mappings_001.json                      - Mappings (first 50K)")
print("  mappings_002.json                      - Mappings (next 50K)")
print("  ...                                    - Additional mapping chunks as needed")

Generating OCL Concepts from merged_loinc_cui_df...
   Processed 1000/202071 concepts.
   Processed 2000/202071 concepts.
   Processed 3000/202071 concepts.
   Processed 4000/202071 concepts.
   Processed 5000/202071 concepts.
   Processed 6000/202071 concepts.
   Processed 7000/202071 concepts.
   Processed 8000/202071 concepts.
   Processed 9000/202071 concepts.
   Processed 10000/202071 concepts.
   Processed 11000/202071 concepts.
   Processed 12000/202071 concepts.
   Processed 13000/202071 concepts.
   Processed 14000/202071 concepts.
   Processed 15000/202071 concepts.
   Processed 16000/202071 concepts.
   Processed 17000/202071 concepts.
   Processed 18000/202071 concepts.
   Processed 19000/202071 concepts.
   Processed 20000/202071 concepts.
   Processed 21000/202071 concepts.
   Processed 22000/202071 concepts.
   Processed 23000/202071 concepts.
   Processed 24000/202071 concepts.
   Processed 25000/202071 concepts.
   Processed 26000/202071 concepts.
   Processed 27000/20

In [26]:
# OLD # Extract a Small Subset of Related LOINC Concepts and Mappings

# def extract_concept_id_from_url(url):
#     """
#     Extract concept ID from a single URL string.
#     """
#     if not isinstance(url, str) or not url.strip():
#         return None
    
#     try:
#         # URL format: "/orgs/Regenstrief/sources/LOINC/concepts/ID/"
#         parts = url.strip('/').split('/')
#         if len(parts) >= 5 and parts[-2] == 'concepts':
#             return parts[-1]
#     except:
#         pass
#     return None

# def get_children_of_concept_optimized(concepts_df, parent_id, max_children=10):
#     """
#     Optimized: Get direct children of a concept using vectorized operations.
#     """
#     children = []
    
#     # Use vectorized string operations for better performance
#     mask = concepts_df['parent_concept_urls'].notna()
#     potential_children = concepts_df[mask]
    
#     for idx, row in potential_children.head(min(1000, len(potential_children))).iterrows():
#         parent_urls = row['parent_concept_urls']
        
#         # Handle different types of parent_urls
#         if isinstance(parent_urls, list):
#             parent_ids = [extract_concept_id_from_url(url) for url in parent_urls]
#         elif isinstance(parent_urls, str):
#             parent_ids = [extract_concept_id_from_url(parent_urls)]
#         else:
#             continue
            
#         if parent_id in parent_ids:
#             children.append(row['id'])
#             if len(children) >= max_children:
#                 break
                
#     return children

# def extract_targeted_loinc_subset(concepts_df, mappings_list, max_concepts=100, max_mappings=50):
#     """
#     FIXED: Extract a focused subset with guaranteed mappings.
#     """
#     print("🔍 Starting optimized LOINC subset extraction...")
#     print(f"📊 Total available: {len(concepts_df):,} concepts, {len(mappings_list):,} mappings")
    
#     selected_concept_ids = set()
    
#     # Step 1: Add essential container concepts
#     container_ids = ['ROOT', 'LOINC_CONTAINER', 'LOINC_PARTS_CONTAINER', 
#                     'ANSWER_LISTS_CONTAINER', 'LOINC_ANSWERS_CONTAINER']
    
#     for container_id in container_ids:
#         if container_id in concepts_df['id'].values:
#             selected_concept_ids.add(container_id)
    
#     print(f"📦 Added {len(selected_concept_ids)} container concepts")
    
#     # Step 2: Add some LOINC codes (focusing on common ones)
#     loinc_concepts = concepts_df[concepts_df['concept_class'] == 'LOINC'].head(20)
#     loinc_ids = loinc_concepts['id'].tolist()
#     selected_concept_ids.update(loinc_ids[:10])  # Add first 10 LOINC codes
    
#     print(f"🧪 Added {min(10, len(loinc_ids))} LOINC codes")
    
#     # Step 3: Add some Answer Lists and their Answers
#     answer_lists = concepts_df[concepts_df['concept_class'] == 'Answer List'].head(5)
#     selected_concept_ids.update(answer_lists['id'].tolist())
    
#     # Add answers for these lists
#     for al_id in answer_lists['id'].tolist():
#         answers = get_children_of_concept_optimized(concepts_df, al_id, 3)
#         selected_concept_ids.update(answers)
    
#     print(f"📝 Added answer lists and their answers")
    
#     # Step 4: Add some LOINC Parts
#     loinc_parts = concepts_df[concepts_df['concept_class'] == 'LOINC Part'].head(15)
#     selected_concept_ids.update(loinc_parts['id'].tolist())
    
#     print(f"🧩 Added LOINC parts")
    
#     # Limit total concepts
#     if len(selected_concept_ids) > max_concepts:
#         selected_concept_ids = set(list(selected_concept_ids)[:max_concepts])
    
#     print(f"📋 Total selected concepts: {len(selected_concept_ids)}")
    
#     # Step 5: Extract selected concepts
#     subset_concepts_df = concepts_df[concepts_df['id'].isin(selected_concept_ids)].copy()
    
#     # Step 6: OPTIMIZED mapping search - convert to set for O(1) lookup
#     print("🔗 Searching for related mappings (optimized)...")
    
#     related_mappings = []
#     selected_ids_set = set(selected_concept_ids)  # Convert to set for faster lookup
    
#     # Create URL patterns to search for
#     url_patterns = [f"/orgs/Regenstrief/sources/LOINC/concepts/{cid}/" for cid in selected_concept_ids]
#     url_patterns_set = set(url_patterns)
    
#     # Search through mappings more efficiently
#     mapping_count = 0
#     for mapping in mappings_list:
#         # Check if this mapping involves any of our selected concepts
#         mapping_involves_selected = False
        
#         # Check from_concept_url
#         from_url = mapping.get('from_concept_url', '')
#         if from_url in url_patterns_set:
#             mapping_involves_selected = True
        
#         # Check to_concept_url
#         to_url = mapping.get('to_concept_url', '')
#         if to_url in url_patterns_set:
#             mapping_involves_selected = True
        
#         # Also check for direct concept code references
#         for field in ['from_concept_code', 'to_concept_code']:
#             if field in mapping and mapping[field] in selected_ids_set:
#                 mapping_involves_selected = True
#                 break
        
#         if mapping_involves_selected:
#             related_mappings.append(mapping)
            
#         mapping_count += 1
        
#         # Progress indicator and early exit
#         if mapping_count % 50000 == 0:
#             print(f"  📊 Searched {mapping_count:,} mappings, found {len(related_mappings)} matches...")
        
#         if len(related_mappings) >= max_mappings:
#             print(f"  ✅ Reached mapping limit ({max_mappings})")
#             break
    
#     print(f"🔗 Found {len(related_mappings)} related mappings after searching {mapping_count:,} total mappings")
    
#     # Summary
#     total_items = len(subset_concepts_df) + len(related_mappings)
#     print(f"\n✅ Subset extraction complete!")
#     print(f"📊 Final subset: {len(subset_concepts_df)} concepts + {len(related_mappings)} mappings = {total_items} total items")
    
#     # Display concept breakdown
#     if not subset_concepts_df.empty:
#         concept_breakdown = subset_concepts_df['concept_class'].value_counts()
#         print(f"\n📋 Concept breakdown by class:")
#         for concept_class, count in concept_breakdown.items():
#             print(f"   {concept_class}: {count}")
    
#     # Display mapping breakdown
#     if related_mappings:
#         mapping_types = {}
#         for mapping in related_mappings:
#             map_type = mapping.get('map_type', 'Unknown')
#             mapping_types[map_type] = mapping_types.get(map_type, 0) + 1
        
#         print(f"\n🔗 Mapping breakdown by type:")
#         for map_type, count in mapping_types.items():
#             print(f"   {map_type}: {count}")
    
#     return subset_concepts_df, related_mappings

# # Execute the optimized subset extraction
# try:
#     print("Starting subset extraction...")
    
#     # Use optimized parameters for speed
#     subset_concepts_df, subset_mappings = extract_targeted_loinc_subset(
#         merged_loinc_cui_df, 
#         all_mappings, 
#         max_concepts=100,  # Smaller subset for speed
#         max_mappings=200   # Guaranteed to find some mappings
#     )
    
#     print(f"\n🎯 Subset ready for use!")
#     print(f"   - subset_concepts_df: {len(subset_concepts_df)} concepts")
#     print(f"   - subset_mappings: {len(subset_mappings)} mappings")
    
#     # Quick validation
#     if len(subset_mappings) == 0:
#         print("⚠️  Still no mappings found. Let's debug the mapping structure...")
        
#         # Debug: Check a few sample mappings
#         sample_mappings = all_mappings[:3]
#         print("\n🔍 Sample mapping structure:")
#         for i, mapping in enumerate(sample_mappings):
#             print(f"  Mapping {i+1} keys: {list(mapping.keys())}")
#             if 'from_concept_url' in mapping:
#                 print(f"    from_concept_url: {mapping['from_concept_url']}")
#             if 'to_concept_url' in mapping:
#                 print(f"    to_concept_url: {mapping['to_concept_url']}")
    
#     print(f"\n💾 Subset extraction completed successfully!")
    
    
# except NameError as e:
#     print(f"❌ Error: Required variables not found: {e}")
#     print("   Make sure merged_loinc_cui_df and all_mappings exist")
# except Exception as e:
#     print(f"❌ Error during subset extraction: {e}")
#     import traceback
#     traceback.print_exc()

In [27]:
# OLD # Continue from the subset extraction cell above...

# # Output Subset to OCL-formatted JSON Files
# def write_subset_to_ocl_format(concepts_df, mappings_list, output_prefix="subset"):
#     """
#     Write the extracted subset to OCL-formatted JSON files.
#     Reuses the same formatting and cleaning functions from the main output generation.
#     """
#     print(f"\n📤 Writing subset to OCL-formatted JSON files...")
    
#     # Process subset concepts
#     if not concepts_df.empty:
#         print(f"🔄 Processing {len(concepts_df)} subset concepts...")
        
#         concepts_data = []
#         for idx, row in concepts_df.iterrows():
#             row_dict = row.to_dict()
            
#             # Logic to dynamically process all 'names' attributes with locale_preferred flag
#             names_list = []
#             keys_to_remove = []
#             for key, value in list(row_dict.items()):
#                 if key.startswith('names.'):
#                     if is_valid_value(value):
#                         # Use regex to extract name_type and locale
#                         match = re.match(r'names\.([^.]+)\.([^\[]+)\[\d+\]', key)
#                         if match:
#                             name_type, locale = match.groups()
#                             locale_preferred = (name_type == 'Fully-Specified')
#                             names_list.append({
#                                 'name': value, 
#                                 'name_type': name_type, 
#                                 'locale': locale, 
#                                 'locale_preferred': locale_preferred
#                             })
#                     del row_dict[key]
            
#             if names_list:
#                 row_dict['names'] = names_list
            
#             # Handle description
#             description = row_dict.pop('description', None)
#             if is_valid_value(description):
#                 row_dict['descriptions'] = [{
#                     'description': description, 
#                     'locale': 'en', 
#                     'description_type': 'Full'
#                 }]
#             else:
#                 row_dict['descriptions'] = []

#             # Logic to handle the 'extras' attribute as a single dictionary
#             extras_dict = {}
#             for key, value in list(row_dict.items()):
#                 if key.startswith('extras.'):
#                     if is_valid_value(value):
#                         # Clean the key name by removing 'extras.' prefix
#                         new_key = key[len('extras.'):]
#                         extras_dict[new_key] = value
#                     del row_dict[key]
            
#             if extras_dict:
#                 row_dict['extras'] = extras_dict

#             # Final cleanup of any remaining invalid values
#             cleaned_row_dict = {k: v for k, v in row_dict.items() if is_valid_value(v)}
#             if 'extras' in cleaned_row_dict and not cleaned_row_dict['extras']:
#                 del cleaned_row_dict['extras']

#             concepts_data.append(cleaned_row_dict)
        
#         # Write concepts to file
#         concepts_filename = f"{output_prefix}_concepts.json"
#         concepts_filepath = os.path.join(output_folder_path, concepts_filename)
        
#         print(f"💾 Writing {len(concepts_data)} concepts to {concepts_filepath}...")
#         with open(concepts_filepath, 'w', encoding='utf-8') as f:
#             for record in concepts_data:
#                 cleaned_record = clean_concept_for_json(record)
#                 json.dump(cleaned_record, f, ensure_ascii=False)
#                 f.write('\n')
        
#         print(f"✅ Concepts written successfully!")
    
#     # Process subset mappings
#     if mappings_list:
#         print(f"🔄 Processing {len(mappings_list)} subset mappings...")
        
#         mappings_data = []
#         for mapping in mappings_list:
#             mapping_dict = mapping.copy()

#             # Logic to handle the 'extras' attribute as a single dictionary
#             extras_dict = {}
#             for key, value in list(mapping_dict.items()):
#                 if key.startswith('extras.'):
#                     if is_valid_value(value):
#                         # Clean the key name by removing 'extras.' prefix
#                         new_key = key[len('extras.'):]
#                         extras_dict[new_key] = value
#                     del mapping_dict[key]
            
#             if extras_dict:
#                 mapping_dict['extras'] = extras_dict

#             # Final cleanup of any remaining invalid values
#             cleaned_mapping_dict = {k: v for k, v in mapping_dict.items() if is_valid_value(v)}
#             if 'extras' in cleaned_mapping_dict and not cleaned_mapping_dict['extras']:
#                 del cleaned_mapping_dict['extras']
                
#             mappings_data.append(cleaned_mapping_dict)
        
#         # Write mappings to file
#         mappings_filename = f"{output_prefix}_mappings.json"
#         mappings_filepath = os.path.join(output_folder_path, mappings_filename)
        
#         print(f"💾 Writing {len(mappings_data)} mappings to {mappings_filepath}...")
#         with open(mappings_filepath, 'w', encoding='utf-8') as f:
#             for record in mappings_data:
#                 cleaned_record = clean_concept_for_json(record)
#                 json.dump(cleaned_record, f, ensure_ascii=False)
#                 f.write('\n')
        
#         print(f"✅ Mappings written successfully!")
    
#     # Create a subset summary file
#     summary_filename = f"{output_prefix}_summary.json"
#     summary_filepath = os.path.join(output_folder_path, summary_filename)
    
#     # Generate concept class breakdown
#     concept_breakdown = {}
#     if not concepts_df.empty:
#         concept_breakdown = concepts_df['concept_class'].value_counts().to_dict()
    
#     # Generate mapping type breakdown
#     mapping_breakdown = {}
#     if mappings_list:
#         for mapping in mappings_list:
#             map_type = mapping.get('map_type', 'Unknown')
#             mapping_breakdown[map_type] = mapping_breakdown.get(map_type, 0) + 1
    
#     summary_data = {
#         "subset_info": {
#             "extraction_timestamp": pd.Timestamp.now().isoformat(),
#             "total_concepts": len(concepts_df) if not concepts_df.empty else 0,
#             "total_mappings": len(mappings_list) if mappings_list else 0,
#             "output_files": {
#                 "concepts": concepts_filename if not concepts_df.empty else None,
#                 "mappings": mappings_filename if mappings_list else None,
#                 "summary": summary_filename
#             }
#         },
#         "concept_breakdown": concept_breakdown,
#         "mapping_breakdown": mapping_breakdown,
#         "sample_concepts": concepts_df[['id', 'concept_class']].head(10).to_dict('records') if not concepts_df.empty else [],
#         "sample_mappings": [
#             {k: v for k, v in mapping.items() if k in ['map_type', 'from_concept_url', 'to_concept_url']}
#             for mapping in mappings_list[:5]
#         ] if mappings_list else []
#     }
    
#     print(f"💾 Writing summary to {summary_filepath}...")
#     with open(summary_filepath, 'w', encoding='utf-8') as f:
#         json.dump(summary_data, f, ensure_ascii=False, indent=2)
    
#     print(f"✅ Summary written successfully!")
    
#     # Final summary
#     total_files = 1  # summary file
#     if not concepts_df.empty:
#         total_files += 1
#     if mappings_list:
#         total_files += 1
    
#     print(f"\n🎉 Subset output complete!")
#     print(f"📁 Generated {total_files} files in '{output_folder_path}' directory:")
#     if not concepts_df.empty:
#         print(f"   📄 {concepts_filename} - {len(concepts_df)} concepts in OCL JSON Lines format")
#     if mappings_list:
#         print(f"   📄 {mappings_filename} - {len(mappings_list)} mappings in OCL JSON Lines format")
#     print(f"   📄 {summary_filename} - Subset summary and metadata")
    
#     return {
#         'concepts_file': concepts_filepath if not concepts_df.empty else None,
#         'mappings_file': mappings_filepath if mappings_list else None,
#         'summary_file': summary_filepath
#     }

# # Execute the subset output generation
# try:
#     # Ensure we have the subset data from the previous extraction
#     if 'subset_concepts_df' in locals() and 'subset_mappings' in locals():
#         output_files = write_subset_to_ocl_format(
#             subset_concepts_df, 
#             subset_mappings, 
#             output_prefix="loinc_subset"
#         )
        
#         print(f"\n🔍 Quick validation of generated files:")
#         for file_type, filepath in output_files.items():
#             if filepath and os.path.exists(filepath):
#                 file_size = os.path.getsize(filepath)
#                 print(f"   ✅ {file_type}: {filepath} ({file_size:,} bytes)")
#             elif filepath:
#                 print(f"   ❌ {file_type}: {filepath} (file not found)")
        
#         print(f"\n🚀 Subset files are ready for OCL import!")
#         print(f"   You can now import these files into your OCL instance for testing.")
#         print(f"   The files follow the same OCL Bulk Import JSON Lines format as the full dataset.")
        
#     else:
#         print("❌ Error: Subset data not found.")
#         print("   Please run the subset extraction cell first to generate:")
#         print("   - subset_concepts_df")
#         print("   - subset_mappings")
        
# except Exception as e:
#     print(f"❌ Error during subset output generation: {e}")
#     import traceback
#     traceback.print_exc()

# print(f"\n📊 Final Summary:")
# print(f"   🏗️  Full dataset: {len(merged_loinc_cui_df):,} concepts, {len(all_mappings):,} mappings")
# if 'subset_concepts_df' in locals() and 'subset_mappings' in locals():
#     print(f"   🧪 Test subset: {len(subset_concepts_df):,} concepts, {len(subset_mappings):,} mappings")
#     print(f"   📈 Subset represents {(len(subset_concepts_df)/len(merged_loinc_cui_df)*100):.2f}% of concepts")
#     print(f"   📈 Subset represents {(len(subset_mappings)/len(all_mappings)*100):.2f}% of mappings")

# print(f"\n✅ LOINC to OCL transformation complete!")
# print(f"   All output files are available in the '{output_folder}' directory.")

In [ ]:
# Extract a Small Subset of Related LOINC Concepts and Mappings (including hierarchy file)

def extract_concept_id_from_url(url):
    """
    Extract concept ID from a single URL string.
    """
    if not isinstance(url, str) or not url.strip():
        return None
    
    try:
        # URL format: "/orgs/Regenstrief/sources/LOINC/concepts/ID/"
        parts = url.strip('/').split('/')
        if len(parts) >= 5 and parts[-2] == 'concepts':
            return parts[-1]
    except:
        pass
    return None

def get_children_of_concept_optimized(concepts_df, parent_id, max_children=10):
    """
    Optimized: Get direct children of a concept using vectorized operations.
    """
    children = []
    
    # Use vectorized string operations for better performance
    mask = concepts_df['parent_concept_urls'].notna()
    potential_children = concepts_df[mask]
    
    for idx, row in potential_children.head(min(1000, len(potential_children))).iterrows():
        parent_urls = row['parent_concept_urls']
        
        # Handle different types of parent_urls
        if isinstance(parent_urls, list):
            parent_ids = [extract_concept_id_from_url(url) for url in parent_urls]
        elif isinstance(parent_urls, str):
            parent_ids = [extract_concept_id_from_url(parent_urls)]
        else:
            continue
            
        if parent_id in parent_ids:
            children.append(row['id'])
            if len(children) >= max_children:
                break
                
    return children

def extract_targeted_loinc_subset(concepts_df, mappings_list, max_concepts=100, max_mappings=50):
    """
    FIXED: Extract a focused subset with guaranteed mappings.
    """
    print("🔍 Starting optimized LOINC subset extraction...")
    print(f"📊 Total available: {len(concepts_df):,} concepts, {len(mappings_list):,} mappings")
    
    selected_concept_ids = set()
    
    # Step 1: Add essential container concepts
    container_ids = ['ROOT', 'LOINC_CONTAINER', 'LOINC_PARTS_CONTAINER', 
                    'ANSWER_LISTS_CONTAINER', 'LOINC_ANSWERS_CONTAINER']
    
    for container_id in container_ids:
        if container_id in concepts_df['id'].values:
            selected_concept_ids.add(container_id)
    
    print(f"📦 Added {len(selected_concept_ids)} container concepts")
    
    # Step 2: Add some LOINC codes (focusing on common ones)
    loinc_concepts = concepts_df[concepts_df['concept_class'] == 'LOINC'].head(20)
    loinc_ids = loinc_concepts['id'].tolist()
    selected_concept_ids.update(loinc_ids[:10])  # Add first 10 LOINC codes
    
    print(f"🧪 Added {min(10, len(loinc_ids))} LOINC codes")
    
    # Step 3: Add some Answer Lists and their Answers
    answer_lists = concepts_df[concepts_df['concept_class'] == 'Answer List'].head(5)
    selected_concept_ids.update(answer_lists['id'].tolist())
    
    # Add answers for these lists
    for al_id in answer_lists['id'].tolist():
        answers = get_children_of_concept_optimized(concepts_df, al_id, 3)
        selected_concept_ids.update(answers)
    
    print(f"📝 Added answer lists and their answers")
    
    # Step 4: Add some LOINC Parts
    loinc_parts = concepts_df[concepts_df['concept_class'] == 'LOINC Part'].head(15)
    selected_concept_ids.update(loinc_parts['id'].tolist())
    
    print(f"🧩 Added LOINC parts")
    
    # Limit total concepts
    if len(selected_concept_ids) > max_concepts:
        selected_concept_ids = set(list(selected_concept_ids)[:max_concepts])
    
    print(f"📋 Total selected concepts: {len(selected_concept_ids)}")
    
    # Step 5: Extract selected concepts
    subset_concepts_df = concepts_df[concepts_df['id'].isin(selected_concept_ids)].copy()
    
    # Step 6: OPTIMIZED mapping search - convert to set for O(1) lookup
    print("🔗 Searching for related mappings (optimized)...")
    
    related_mappings = []
    selected_ids_set = set(selected_concept_ids)  # Convert to set for faster lookup
    
    # Create URL patterns to search for
    url_patterns = [f"/orgs/Regenstrief/sources/LOINC/concepts/{cid}/" for cid in selected_concept_ids]
    url_patterns_set = set(url_patterns)
    
    # Search through mappings more efficiently
    mapping_count = 0
    for mapping in mappings_list:
        # Check if this mapping involves any of our selected concepts
        mapping_involves_selected = False
        
        # Check from_concept_url
        from_url = mapping.get('from_concept_url', '')
        if from_url in url_patterns_set:
            mapping_involves_selected = True
        
        # Check to_concept_url
        to_url = mapping.get('to_concept_url', '')
        if to_url in url_patterns_set:
            mapping_involves_selected = True
        
        # Also check for direct concept code references
        for field in ['from_concept_code', 'to_concept_code']:
            if field in mapping and mapping[field] in selected_ids_set:
                mapping_involves_selected = True
                break
        
        if mapping_involves_selected:
            related_mappings.append(mapping)
            
        mapping_count += 1
        
        # Progress indicator and early exit
        if mapping_count % 50000 == 0:
            print(f"  📊 Searched {mapping_count:,} mappings, found {len(related_mappings)} matches...")
        
        if len(related_mappings) >= max_mappings:
            print(f"  ✅ Reached mapping limit ({max_mappings})")
            break
    
    print(f"🔗 Found {len(related_mappings)} related mappings after searching {mapping_count:,} total mappings")
    
    # Summary
    total_items = len(subset_concepts_df) + len(related_mappings)
    print(f"\n✅ Subset extraction complete!")
    print(f"📊 Final subset: {len(subset_concepts_df)} concepts + {len(related_mappings)} mappings = {total_items} total items")
    
    # Display concept breakdown
    if not subset_concepts_df.empty:
        concept_breakdown = subset_concepts_df['concept_class'].value_counts()
        print(f"\n📋 Concept breakdown by class:")
        for concept_class, count in concept_breakdown.items():
            print(f"   {concept_class}: {count}")
    
    # Display mapping breakdown
    if related_mappings:
        mapping_types = {}
        for mapping in related_mappings:
            map_type = mapping.get('map_type', 'Unknown')
            mapping_types[map_type] = mapping_types.get(map_type, 0) + 1
        
        print(f"\n🔗 Mapping breakdown by type:")
        for map_type, count in mapping_types.items():
            print(f"   {map_type}: {count}")
    
    return subset_concepts_df, related_mappings

def build_concept_url(concept_id, org='Regenstrief', source='LOINC'):
    """
    Builds a concept URL from concept ID following OCL URL pattern.
    """
    return f"/orgs/{org}/sources/{source}/concepts/{concept_id}/"

def build_hierarchy_mapping(data_df):
    """
    Builds a hierarchy mapping dictionary where keys are parent concept URLs
    and values are lists of child concept URLs.
    """
    print("Building hierarchy mapping...")
    
    hierarchy_map = {}
    total_concepts = len(data_df)
    concepts_with_parents = 0
    
    for idx, row in data_df.iterrows():
        if (idx + 1) % 100 == 0 or (idx + 1) == total_concepts:
            print(f"   Processed {idx + 1}/{total_concepts} concepts for hierarchy.")
        
        concept_id = row.get('id')
        if not concept_id:
            continue
            
        # Build the current concept's URL
        child_url = build_concept_url(concept_id)
        
        # Get parent URLs
        parent_urls = row.get('parent_concept_urls', [])
        
        # Handle case where parent_urls might be NaN or None
        if parent_urls is None or (hasattr(parent_urls, '__len__') and len(parent_urls) == 0):
            continue
            
        # Ensure parent_urls is a list
        if not isinstance(parent_urls, (list, tuple)):
            try:
                if pd.isna(parent_urls):
                    continue
            except (TypeError, ValueError):
                pass
            # If it's a single URL string, convert to list
            if isinstance(parent_urls, str):
                parent_urls = [parent_urls]
            else:
                continue
        
        # Add this concept as a child to each of its parents
        for parent_url in parent_urls:
            if parent_url and isinstance(parent_url, str) and parent_url.strip():
                parent_url = parent_url.strip()
                
                # Initialize parent entry if it doesn't exist
                if parent_url not in hierarchy_map:
                    hierarchy_map[parent_url] = []
                
                # Add child URL if not already present
                if child_url not in hierarchy_map[parent_url]:
                    hierarchy_map[parent_url].append(child_url)
        
        if parent_urls:
            concepts_with_parents += 1
    
    print(f"Hierarchy mapping complete:")
    print(f"  - Total concepts processed: {total_concepts:,}")
    print(f"  - Concepts with parents: {concepts_with_parents:,}")
    print(f"  - Parent concepts in hierarchy: {len(hierarchy_map):,}")
    
    # Show some statistics
    child_counts = [len(children) for children in hierarchy_map.values()]
    if child_counts:
        print(f"  - Average children per parent: {sum(child_counts) / len(child_counts):.1f}")
        print(f"  - Max children for a parent: {max(child_counts):,}")
        print(f"  - Total parent-child relationships: {sum(child_counts):,}")
    
    return hierarchy_map

def write_hierarchy_json(hierarchy_map, output_filename='hierarchy_only.json'):
    """
    Writes the hierarchy mapping to a JSON file.
    """
    output_file_path = os.path.join(output_folder_path, output_filename)
    
    print(f"Writing hierarchy-only JSON to {output_file_path}...")
    
    # Sort the hierarchy map by parent URL for consistent output
    sorted_hierarchy = dict(sorted(hierarchy_map.items()))
    
    # Also sort the child lists for each parent
    for parent_url in sorted_hierarchy:
        sorted_hierarchy[parent_url] = sorted(sorted_hierarchy[parent_url])
    
    # Write to JSON file with proper encoding
    with open(output_file_path, 'w', encoding='utf-8') as f:
        json.dump(sorted_hierarchy, f, ensure_ascii=False, indent=2)
    
    print(f"Hierarchy-only JSON written successfully!")
    print(f"File size: {os.path.getsize(output_file_path):,} bytes")

# Execute the optimized subset extraction
try:
    print("Starting subset extraction...")
    
    # Use optimized parameters for speed
    subset_concepts_df, subset_mappings = extract_targeted_loinc_subset(
        merged_loinc_cui_df, 
        all_mappings, 
        max_concepts=100,  # Smaller subset for speed
        max_mappings=200   # Guaranteed to find some mappings
    )
    
    print(f"\n🎯 Subset ready for use!")
    print(f"   - subset_concepts_df: {len(subset_concepts_df)} concepts")
    print(f"   - subset_mappings: {len(subset_mappings)} mappings")
    
    # Quick validation
    if len(subset_mappings) == 0:
        print("⚠️  Still no mappings found. Let's debug the mapping structure...")
        
        # Debug: Check a few sample mappings
        sample_mappings = all_mappings[:3]
        print("\n🔍 Sample mapping structure:")
        for i, mapping in enumerate(sample_mappings):
            print(f"  Mapping {i+1} keys: {list(mapping.keys())}")
            if 'from_concept_url' in mapping:
                print(f"    from_concept_url: {mapping['from_concept_url']}")
            if 'to_concept_url' in mapping:
                print(f"    to_concept_url: {mapping['to_concept_url']}")
    
    print(f"\n💾 Subset extraction completed successfully!")
    
    # === NEW: PROCESS SUBSET WITH HIERARCHY ===
    # Apply hierarchy-only processing to the subset
    if len(subset_concepts_df) > 0:
        print("\n" + "="*80)
        print("🌳 PROCESSING SUBSET HIERARCHY")
        print("="*80)
        
        # Build hierarchy mapping for the subset
        subset_hierarchy = build_hierarchy_mapping(subset_concepts_df)
        
        # Write subset hierarchy JSON
        subset_hierarchy_filename = 'subset_hierarchy_only.json'
        write_hierarchy_json(subset_hierarchy, subset_hierarchy_filename)
        
        # Also write the subset concepts and mappings as regular OCL format for comparison
        print(f"\n📝 Writing subset OCL format files...")
        
        # Process subset concepts
        subset_concepts_data = []
        for idx, row in subset_concepts_df.iterrows():
            row_dict = row.to_dict()
            
            # Logic to dynamically process all 'names' attributes with locale_preferred flag
            names_list = []
            for key, value in list(row_dict.items()):
                if key.startswith('names.'):
                    if is_valid_value(value):
                        # Use regex to extract name_type and locale
                        match = re.match(r'names\.([^.]+)\.([^\[]+)\[\d+\]', key)
                        if match:
                            name_type, locale = match.groups()
                            locale_preferred = (name_type == 'Long Common')
                            names_list.append({'name': value, 'name_type': name_type, 'locale': locale, 'locale_preferred': locale_preferred})
                    del row_dict[key]
            
            if names_list:
                row_dict['names'] = names_list
            
            # Handle description
            description = row_dict.pop('description', None)
            if is_valid_value(description):
                row_dict['descriptions'] = [{'description': description, 'locale': 'en', 'description_type': 'Full'}]
            else:
                row_dict['descriptions'] = []

            # Handle extras
            extras_dict = {}
            for key, value in list(row_dict.items()):
                if key.startswith('extras.'):
                    if is_valid_value(value):
                        new_key = key[len('extras.'):]
                        extras_dict[new_key] = value
                    del row_dict[key]
            
            if extras_dict:
                row_dict['extras'] = extras_dict

            # Final cleanup
            cleaned_row_dict = {k: v for k, v in row_dict.items() if is_valid_value(v)}
            if 'extras' in cleaned_row_dict and not cleaned_row_dict['extras']:
                del cleaned_row_dict['extras']

            subset_concepts_data.append(cleaned_row_dict)
        
        # Write subset concepts
        subset_concepts_filename = 'subset_concepts.json'
        subset_concepts_path = os.path.join(output_folder_path, subset_concepts_filename)
        with open(subset_concepts_path, 'w', encoding='utf-8') as f:
            for record in subset_concepts_data:
                cleaned_record = clean_concept_for_json(record)
                json.dump(cleaned_record, f, ensure_ascii=False)
                f.write('\n')
        
        # Process subset mappings
        subset_mappings_data = []
        for mapping in subset_mappings:
            # Handle extras for mappings
            mapping_dict = mapping.copy()
            extras_dict = {}
            for key, value in list(mapping_dict.items()):
                if key.startswith('extras.'):
                    if is_valid_value(value):
                        new_key = key[len('extras.'):]
                        extras_dict[new_key] = value
                    del mapping_dict[key]
            
            if extras_dict:
                mapping_dict['extras'] = extras_dict

            # Final cleanup
            cleaned_mapping_dict = {k: v for k, v in mapping_dict.items() if is_valid_value(v)}
            if 'extras' in cleaned_mapping_dict and not cleaned_mapping_dict['extras']:
                del cleaned_mapping_dict['extras']
                
            subset_mappings_data.append(cleaned_mapping_dict)
        
        # Write subset mappings
        if subset_mappings_data:
            subset_mappings_filename = 'subset_mappings.json'
            subset_mappings_path = os.path.join(output_folder_path, subset_mappings_filename)
            with open(subset_mappings_path, 'w', encoding='utf-8') as f:
                for record in subset_mappings_data:
                    cleaned_record = clean_concept_for_json(record)
                    json.dump(cleaned_record, f, ensure_ascii=False)
                    f.write('\n')
        
        # Summary
        print(f"\n✅ Subset processing complete!")
        print(f"📁 Generated subset files:")
        print(f"   • {subset_hierarchy_filename} - Hierarchy-only JSON ({len(subset_hierarchy)} parent-child relationships)")
        print(f"   • {subset_concepts_filename} - OCL concepts ({len(subset_concepts_data)} concepts)")
        if subset_mappings_data:
            print(f"   • subset_mappings.json - OCL mappings ({len(subset_mappings_data)} mappings)")
        
        # Show hierarchy sample
        if subset_hierarchy:
            print(f"\n🌳 Sample hierarchy relationships from subset:")
            sample_count = 0
            for parent_url, child_urls in subset_hierarchy.items():
                if sample_count >= 3:
                    break
                print(f"   {parent_url}")
                for child_url in child_urls[:3]:  # Show max 3 children per parent
                    print(f"     └─ {child_url}")
                if len(child_urls) > 3:
                    print(f"     └─ ... and {len(child_urls) - 3} more children")
                sample_count += 1
            
            if len(subset_hierarchy) > 3:
                print(f"   ... and {len(subset_hierarchy) - 3} more parent concepts")
        else:
            print(f"\n📝 No hierarchical relationships found in this subset.")
    
except NameError as e:
    print(f"❌ Error: Required variables not found: {e}")
    print("   Make sure merged_loinc_cui_df and all_mappings exist")
except Exception as e:
    print(f"❌ Error during subset extraction: {e}")
    import traceback
    traceback.print_exc()

Starting subset extraction...
🔍 Starting optimized LOINC subset extraction...
📊 Total available: 202,071 concepts, 259,932 mappings
📦 Added 5 container concepts
🧪 Added 10 LOINC codes
📝 Added answer lists and their answers
🧩 Added LOINC parts
📋 Total selected concepts: 35
🔗 Searching for related mappings (optimized)...
  📊 Searched 50,000 mappings, found 1 matches...
  📊 Searched 100,000 mappings, found 16 matches...
  📊 Searched 150,000 mappings, found 16 matches...
  📊 Searched 200,000 mappings, found 16 matches...
  📊 Searched 250,000 mappings, found 16 matches...
🔗 Found 16 related mappings after searching 259,932 total mappings

✅ Subset extraction complete!
📊 Final subset: 35 concepts + 16 mappings = 51 total items

📋 Concept breakdown by class:
   LOINC Part: 15
   LOINC: 10
   Answer List: 5
   Container: 4
   Root: 1

🔗 Mapping breakdown by type:
   Has Element: 4
   Has Answer: 12

🎯 Subset ready for use!
   - subset_concepts_df: 35 concepts
   - subset_mappings: 16 mappings


In [29]:
# Optional - Hierarchy-Only Output - Generate parent-child concept URL relationships

hierarchy_only = True  # Variable to enable hierarchy-only mode

def build_concept_url(concept_id, org='Regenstrief', source='LOINC'):
    """
    Builds a concept URL from concept ID following OCL URL pattern.
    
    Args:
        concept_id: The concept ID
        org: Organization name (default: 'Regenstrief')
        source: Source name (default: 'LOINC')
        
    Returns:
        str: Full concept URL
    """
    return f"/orgs/{org}/sources/{source}/concepts/{concept_id}/"

def build_hierarchy_mapping(data_df):
    """
    Builds a hierarchy mapping dictionary where keys are parent concept URLs
    and values are lists of child concept URLs.
    
    Args:
        data_df: DataFrame containing concept data with 'id' and 'parent_concept_urls' columns
        
    Returns:
        dict: Parent URL -> [Child URLs] mapping
    """
    print("Building hierarchy mapping...")
    
    hierarchy_map = {}
    total_concepts = len(data_df)
    concepts_with_parents = 0
    
    for idx, row in data_df.iterrows():
        if (idx + 1) % 1000 == 0 or (idx + 1) == total_concepts:
            print(f"   Processed {idx + 1}/{total_concepts} concepts for hierarchy.")
        
        concept_id = row.get('id')
        if not concept_id:
            continue
            
        # Build the current concept's URL
        child_url = build_concept_url(concept_id)
        
        # Get parent URLs
        parent_urls = row.get('parent_concept_urls', [])
        
        # Handle case where parent_urls might be NaN or None
        if parent_urls is None or (hasattr(parent_urls, '__len__') and len(parent_urls) == 0):
            continue
            
        # Ensure parent_urls is a list
        if not isinstance(parent_urls, (list, tuple)):
            try:
                if pd.isna(parent_urls):
                    continue
            except (TypeError, ValueError):
                pass
            # If it's a single URL string, convert to list
            if isinstance(parent_urls, str):
                parent_urls = [parent_urls]
            else:
                continue
        
        # Add this concept as a child to each of its parents
        for parent_url in parent_urls:
            if parent_url and isinstance(parent_url, str) and parent_url.strip():
                parent_url = parent_url.strip()
                
                # Initialize parent entry if it doesn't exist
                if parent_url not in hierarchy_map:
                    hierarchy_map[parent_url] = []
                
                # Add child URL if not already present
                if child_url not in hierarchy_map[parent_url]:
                    hierarchy_map[parent_url].append(child_url)
        
        if parent_urls:
            concepts_with_parents += 1
    
    print(f"Hierarchy mapping complete:")
    print(f"  - Total concepts processed: {total_concepts:,}")
    print(f"  - Concepts with parents: {concepts_with_parents:,}")
    print(f"  - Parent concepts in hierarchy: {len(hierarchy_map):,}")
    
    # Show some statistics
    child_counts = [len(children) for children in hierarchy_map.values()]
    if child_counts:
        print(f"  - Average children per parent: {sum(child_counts) / len(child_counts):.1f}")
        print(f"  - Max children for a parent: {max(child_counts):,}")
        print(f"  - Total parent-child relationships: {sum(child_counts):,}")
    
    return hierarchy_map

def write_hierarchy_json(hierarchy_map, output_filename='hierarchy_only.json'):
    """
    Writes the hierarchy mapping to a JSON file.
    
    Args:
        hierarchy_map: Dictionary with parent URLs as keys and child URL lists as values
        output_filename: Name of the output file
    """
    output_file_path = os.path.join(output_folder_path, output_filename)
    
    print(f"Writing hierarchy-only JSON to {output_file_path}...")
    
    # Sort the hierarchy map by parent URL for consistent output
    sorted_hierarchy = dict(sorted(hierarchy_map.items()))
    
    # Also sort the child lists for each parent
    for parent_url in sorted_hierarchy:
        sorted_hierarchy[parent_url] = sorted(sorted_hierarchy[parent_url])
    
    # Write to JSON file with proper encoding
    with open(output_file_path, 'w', encoding='utf-8') as f:
        json.dump(sorted_hierarchy, f, ensure_ascii=False, indent=2)
    
    print(f"Hierarchy-only JSON written successfully!")
    print(f"File size: {os.path.getsize(output_file_path):,} bytes")

# Execute hierarchy-only processing if enabled
if hierarchy_only:
    print("=" * 60)
    print("HIERARCHY-ONLY MODE ENABLED")
    print("=" * 60)
    
    # Ensure output folder exists
    os.makedirs(output_folder_path, exist_ok=True)
    
    # Build the hierarchy mapping
    hierarchy_mapping = build_hierarchy_mapping(merged_loinc_cui_df)
    
    # Write to JSON file
    write_hierarchy_json(hierarchy_mapping, 'hierarchy_only.json')
    
    # Show a sample of the output
    print("\nSample hierarchy relationships:")
    sample_count = 0
    for parent_url, child_urls in hierarchy_mapping.items():
        if sample_count >= 3:  # Show first 3 examples
            break
        print(f"  {parent_url}: {child_urls}")
        sample_count += 1
    
    if len(hierarchy_mapping) > 3:
        print(f"  ... and {len(hierarchy_mapping) - 3} more parent-child relationships")
    
    print(f"\nHierarchy-only processing complete!")
    print(f"Output file: {os.path.join(output_folder_path, 'hierarchy_only.json')}")

else:
    print("Hierarchy-only mode is disabled. Set hierarchy_only = True to enable.")

HIERARCHY-ONLY MODE ENABLED
Building hierarchy mapping...
   Processed 1000/202071 concepts for hierarchy.
   Processed 2000/202071 concepts for hierarchy.
   Processed 3000/202071 concepts for hierarchy.
   Processed 4000/202071 concepts for hierarchy.
   Processed 5000/202071 concepts for hierarchy.
   Processed 6000/202071 concepts for hierarchy.
   Processed 7000/202071 concepts for hierarchy.
   Processed 8000/202071 concepts for hierarchy.
   Processed 9000/202071 concepts for hierarchy.
   Processed 10000/202071 concepts for hierarchy.
   Processed 11000/202071 concepts for hierarchy.
   Processed 12000/202071 concepts for hierarchy.
   Processed 13000/202071 concepts for hierarchy.
   Processed 14000/202071 concepts for hierarchy.
   Processed 15000/202071 concepts for hierarchy.
   Processed 16000/202071 concepts for hierarchy.
   Processed 17000/202071 concepts for hierarchy.
   Processed 18000/202071 concepts for hierarchy.
   Processed 19000/202071 concepts for hierarchy.
 